# Notebook 05 — Domain-Matched KG Construction and Ablation Rerun

This notebook constructs a domain-matched knowledge graph (Phase C) from entity
surface strings extracted directly from the Adkins et al. (2025) corpus, trains
TransE embeddings on the resulting triples, and reruns the full five-condition
ablation study under guaranteed 100% surface string coverage.

Notebooks 00–04 established two candidate explanations for the aggregate null
result: (1) coverage failure — 60.7% of test entities are absent from the Phase
A and Phase B KGs; (2) signal quality failure — even covered entities show a
small consistent degradation relative to no-KG on identical sentence subsets
(Notebook 04, corrected comparison). Phase C eliminates the coverage problem
by construction, isolating signal quality as the sole remaining variable.

The experiment does not constitute data leakage. Only entity surface strings are
extracted from the test split; labels are not used and the test CoNLL file is
not loaded during training or KG construction. The KG encodes relational
structure (type, country, party, employer, location) and co-occurrence
proximity, not NER label assignments.

If the five-condition ablation under full coverage reproduces the null result,
the conclusion is definitive: TransE embeddings do not provide a viable
augmentation signal for token-level Irish NER in this architecture, and the
coverage ceiling identified in Notebook 00 was a secondary rather than primary
explanatory mechanism.

**Phase summary**
- Phase A: Wikidata political figures, QID-keyed, parliamentary domain
- Phase B: Oireachtas co-occurrence, surface-keyed, parliamentary domain
- Phase C: Adkins corpus entities, surface-keyed, domain-matched (this notebook)

In [1]:
!pip install vcs-versioning
!pip install seqeval --use-pep517

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16252 sha256=5d52c151201b818997e14aa79de2e861ffa0ccde33fbe830860361ddca94ca8b
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [2]:
# CELL 1 — imports and installation
import subprocess
subprocess.run(['pip', 'install', 'pytorch-crf', 'seqeval', '-q'])
!pip install pytorch-crf

import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import numpy as np
import pandas as pd
import json
import os
import time
import requests
from collections import defaultdict
from torchcrf import CRF
from transformers import AutoTokenizer, AutoModel
from seqeval.metrics import f1_score

print(f'torch:        {torch.__version__}')
print(f'device:       {torch.device("cuda" if torch.cuda.is_available() else "cpu")}')

torch:        2.10.0+cpu
device:       cpu


In [3]:
# CELL 2 — paths and constants
DATA         = '/kaggle/input/datasets/michaelmarkey64/irish-ner-kg-consolidated'
ADKINS       = '/kaggle/input/datasets/michaelmarkey64/adkins-et-al-2025'
CHECKPOINTS  = '/kaggle/input/datasets/michaelmarkey64/irish-ner-model-checkpoints'
ABLATION     = '/kaggle/input/datasets/michaelmarkey64/irish-ner-ablation-results'
WORKING      = '/kaggle/working'

MODEL_NAME   = 'DCU-NLP/bert-base-irish-cased-v1'
MAX_LEN      = 128
BERT_DIM     = 768
KG_DIM       = 128
EPOCHS       = 5
BATCH_SIZE   = 16
LR           = 2e-5
SEEDS        = [42, 123, 256, 512, 999, 1024, 2048]

SPARQL_URL   = 'https://query.wikidata.org/sparql'
SPARQL_UA    = 'IrishNERResearch/1.0'
RATE_LIMIT   = 1.0  # seconds between Wikidata requests

COOC_WINDOW  = 5    # token window for co-occurrence fallback

device = torch.device('cpu')
print(f'Device: {device}')
print(f'KG output path: {WORKING}/phase_c_embeddings.pkl')

Device: cpu
KG output path: /kaggle/working/phase_c_embeddings.pkl


In [4]:
# CELL 3 — CoNLL loader and label vocabulary
def load_conll(path):
    sentences, sentence_labels = [], []
    tokens, labels = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append(tokens)
                    sentence_labels.append(labels)
                    tokens, labels = [], []
            else:
                parts = line.split()
                tokens.append(parts[0])
                labels.append(parts[-1])
    if tokens:
        sentences.append(tokens)
        sentence_labels.append(labels)
    return sentences, sentence_labels

train_tokens, train_labels = load_conll(f'{ADKINS}/train_final.conll')
val_tokens,   val_labels   = load_conll(f'{ADKINS}/NER_Irish_validation.conll')
test_tokens,  test_labels  = load_conll(f'{ADKINS}/NER_Irish_test.conll')

all_labels = sorted(set(l for seq in train_labels for l in seq))
label2id   = {l: i for i, l in enumerate(all_labels)}
id2label   = {i: l for l, i in label2id.items()}

print(f'Train sentences:    {len(train_tokens)}')
print(f'Val sentences:      {len(val_tokens)}')
print(f'Test sentences:     {len(test_tokens)}')
print(f'Label vocabulary:   {all_labels}')

Train sentences:    1006
Val sentences:      100
Test sentences:     140
Label vocabulary:   ['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']


In [5]:
# CELL 4 — entity extraction across all three splits
def get_entity_spans_with_type(tokens, labels):
    entities = []
    current, current_type = [], None
    for token, label in zip(tokens, labels):
        if label.startswith('B-'):
            if current:
                entities.append((' '.join(current), current_type))
            current = [token]
            current_type = label[2:]
        elif label.startswith('I-') and current:
            current.append(token)
        else:
            if current:
                entities.append((' '.join(current), current_type))
            current, current_type = [], None
    if current:
        entities.append((' '.join(current), current_type))
    return entities

def normalise_surface(surface):
    # Strip leading/trailing punctuation artefacts from CoNLL tokenisation
    # Preserves internal punctuation (hyphens, dots) and Irish characters
    return surface.strip("'\"«»„""\u2018\u2019\u201c\u201d")

entity_index = defaultdict(set)

for split_tokens, split_labels in [
    (train_tokens, train_labels),
    (val_tokens,   val_labels),
    (test_tokens,  test_labels),
]:
    for tokens, labels in zip(split_tokens, split_labels):
        for surface, etype in get_entity_spans_with_type(tokens, labels):
            normalised = normalise_surface(surface)
            if normalised:
                entity_index[normalised].add(etype)

TYPE_PRIORITY = {'PER': 0, 'ORG': 1, 'LOC': 2}
entity_list = []
for surface, types in entity_index.items():
    etype = sorted(types, key=lambda t: TYPE_PRIORITY.get(t, 99))[0]
    entity_list.append((surface, etype))

entity_list = sorted(entity_list, key=lambda x: x[0])

per_count = sum(1 for _, t in entity_list if t == 'PER')
org_count = sum(1 for _, t in entity_list if t == 'ORG')
loc_count = sum(1 for _, t in entity_list if t == 'LOC')

print(f'Unique entity surfaces (all splits): {len(entity_list)}')
print(f'  PER: {per_count}')
print(f'  ORG: {org_count}')
print(f'  LOC: {loc_count}')

Unique entity surfaces (all splits): 1863
  PER: 651
  ORG: 654
  LOC: 558


In [6]:
# CELL 5 — Wikidata resolver with English fallback and relation extraction
import re

RELATIONS = {
    'P31':  'instance_of',
    'P17':  'country',
    'P102': 'party',
    'P108': 'employer',
    'P131': 'located_in',
}

DISAMBIGUATION_QIDS = {'Q4167410', 'Q22808320', 'Q227699'}

def make_query(label, lang):
    prop_selects = ' '.join(f'?prop{pid}Label' for pid in RELATIONS)
    return (
        f'SELECT ?item ?itemLabel ?itemDescription {prop_selects} WHERE {{\n'
        f'  ?item rdfs:label "{label}"@{lang} .\n'
        f'  OPTIONAL {{ ?item wdt:P31  ?propP31.  }}\n'
        f'  OPTIONAL {{ ?item wdt:P17  ?propP17.  }}\n'
        f'  OPTIONAL {{ ?item wdt:P102 ?propP102. }}\n'
        f'  OPTIONAL {{ ?item wdt:P108 ?propP108. }}\n'
        f'  OPTIONAL {{ ?item wdt:P131 ?propP131. }}\n'
        f'  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ga,en". }}\n'
        f'}}\n'
        f'LIMIT 1'
    )

def generate_irish_candidates(label):
    sentence_case = label[0].upper() + label[1:].lower() if len(label) > 1 else label.upper()
    candidates = [label, sentence_case, label.lower()]

    mutation_pattern = re.compile(r'\b(t|h)([A-ZÁÉÍÓÚÀÈÌÒÙ])', re.UNICODE)

    def fix_mutation(m):
        return m.group(1).lower() + m.group(2).upper()

    mutated = mutation_pattern.sub(fix_mutation, sentence_case)
    if mutated != sentence_case:
        candidates.append(mutated)

    return dict.fromkeys(candidates)

def query_wikidata_sparql_ga(label):
    clean = label.replace(' ', '')
    if len(clean) < 4 or not any(c.isalpha() for c in label):
        return None, None, {}, None

    candidates = generate_irish_candidates(label)

    for lang in ['ga', 'en']:
        for candidate in candidates:
            query = make_query(candidate, lang)
            try:
                r = requests.get(
                    SPARQL_URL,
                    params={'query': query, 'format': 'json'},
                    headers={'User-Agent': SPARQL_UA},
                    timeout=15
                )
                bindings = r.json().get('results', {}).get('bindings', [])
                if bindings:
                    row  = bindings[0]
                    qid  = row['item']['value'].split('/')[-1]
                    if qid in DISAMBIGUATION_QIDS:
                        continue
                    desc = row.get('itemDescription', {}).get('value', '')
                    if 'disambiguation' in desc.lower():
                        continue
                    rels = {}
                    for pid, rel_name in RELATIONS.items():
                        key = 'prop' + pid + 'Label'
                        if key in row:
                            rels[rel_name] = row[key]['value']
                    return qid, desc, rels, lang
            except:
                pass
            time.sleep(RATE_LIMIT)
    return None, None, {}, None

# Smoke test on five entities
print('Smoke test — five entities:')
for surface, etype in entity_list[:5]:
    qid, desc, rels, lang = query_wikidata_sparql_ga(surface)
    if qid:
        print(f'  RESOLVED ({lang}): {surface} [{etype}] -> {qid} — {desc} | rels: {list(rels.keys())}')
    else:
        print(f'  MISSED:           {surface} [{etype}]')
    time.sleep(RATE_LIMIT)

Smoke test — five entities:
  MISSED:           ( Y ) OUR STORY [ORG]
  MISSED:           14 Sráid Henrietta [LOC]
  MISSED:           AE [ORG]
  MISSED:           AIE [ORG]
  MISSED:           AN tAONTAS EORPACH [ORG]


In [7]:
# CELL 6 — load GeoNames Ireland places
import os
import io
import pandas as pd

# The exact verified Kaggle path
GEONAMES_PATH = '/kaggle/input/datasets/michaelmarkey64/geonames-ireland/IE/IE.txt'

# IE.txt columns (tab-separated, no header):
IE_COLS = [
    'geonameid', 'name', 'asciiname', 'alternatenames',
    'lat', 'lng', 'feature_class', 'feature_code',
    'country', 'cc2', 'admin1', 'admin2', 'admin3', 'admin4',
    'population', 'elevation', 'dem', 'timezone', 'modification_date'
]

# Load the dataframe
ie_df = pd.read_csv(
    GEONAMES_PATH,
    sep='\t',
    header=None,
    names=IE_COLS,
    low_memory=False,
    dtype=str
)

print(f'GeoNames Ireland entries: {len(ie_df)}')
print(f'Feature classes: {ie_df["feature_class"].fillna("Unknown").value_counts().to_dict()}')
print(f'\nSample entries:')
print(ie_df[['geonameid', 'name', 'feature_class', 'feature_code']].head(10).to_string(index=False))

GeoNames Ireland entries: 29934
Feature classes: {'P': 12159, 'S': 6721, 'T': 3970, 'L': 3507, 'H': 3241, 'V': 118, 'A': 112, 'R': 106}

Sample entries:
geonameid            name feature_class feature_code
  2635367 Tullyrossmearan             P          PPL
  2649110     River Foyle             H          STM
  2649929      River Erne             H          STM
  2650972   Dromore Lough             H           LK
  2650974    Dromore Head             T         CAPE
  2652852           Cliff             S          EST
  2652973  Claddagh River             H          STM
  2654332        Buncrana             P          PPL
  2960863    Dromore West             P          PPL
  2960864   Dromore House             S          EST


In [8]:
# CELL 7 — load Irish-language alternate names filtered to Ireland
# alternateNamesV2.txt columns:
# 0:alternateNameId 1:geonameid 2:isolanguage 3:alternate_name
# 4:isPreferredName 5:isShortName 6:isColloquial 7:isHistoric
# 8:from 9:to

ALT_COLS = [
    'alternateNameId', 'geonameid', 'isolanguage', 'alternate_name',
    'isPreferredName', 'isShortName', 'isColloquial', 'isHistoric',
    'from', 'to'
]

# Load only Irish-language (ga) alternate names
# Use chunksize to avoid loading the full 130MB file into memory
ireland_geoname_ids = set(ie_df['geonameid'].dropna().unique())

ga_names = []
chunk_size = 100_000

for chunk in pd.read_csv(
    '/kaggle/input/datasets/michaelmarkey64/geonames-ireland/alternateNamesV2/alternateNamesV2.txt',
    sep='\t',
    header=None,
    names=ALT_COLS,
    low_memory=False,
    dtype=str,
    chunksize=chunk_size
):
    filtered = chunk[
        (chunk['isolanguage'] == 'ga') &
        (chunk['geonameid'].isin(ireland_geoname_ids))
    ]
    ga_names.append(filtered)

ga_df = pd.concat(ga_names, ignore_index=True)

print(f'Irish-language alternate names for Ireland: {len(ga_df)}')
print(f'\nSample Irish names:')
print(ga_df[['geonameid', 'alternate_name']].head(10).to_string(index=False))

Irish-language alternate names for Ireland: 2948

Sample Irish names:
geonameid       alternate_name
  2649110           An Feabhal
  2654332         Bun Cranncha
  2960863   An Droim Mór Thiar
  2960866       Cuan Eochaille
  2960869             Eochaill
  2960874       An Aitinn Bhuí
  2960895            An Ghráig
  2960901 an Droichead Adhmaid
  2960909    Bearna na Gaoithe
  2960924           Baile Liam


In [9]:
# CELL 8 — build GeoNames LOC lookup and check overlap with Adkins entities

# Build a lookup: Irish surface string -> (geonameid, english_name, feature_code)
# Priority: Irish alternate name first, English name as fallback

# Map geonameid -> irish alternate names
geoname_to_ga = defaultdict(list)
for _, row in ga_df.iterrows():
    geoname_to_ga[row['geonameid']].append(row['alternate_name'])

# Build the lookup table
geonames_loc_lookup = {}  # surface_string -> {'geonameid', 'english_name', 'feature_code', 'lang'}

for _, row in ie_df.iterrows():
    gid        = row['geonameid']
    eng_name   = row['name']
    feat_code  = row['feature_code']
    feat_class = row['feature_class']

    # Register English name
    if pd.notna(eng_name) and eng_name.strip():
        geonames_loc_lookup[eng_name.strip()] = {
            'geonameid':    gid,
            'english_name': eng_name,
            'feature_code': feat_code,
            'feature_class': feat_class,
            'lang': 'en'
        }

    # Register Irish alternate names (override English if present)
    for ga_name in geoname_to_ga.get(gid, []):
        if ga_name and ga_name.strip():
            geonames_loc_lookup[ga_name.strip()] = {
                'geonameid':    gid,
                'english_name': eng_name,
                'feature_code': feat_code,
                'feature_class': feat_class,
                'lang': 'ga'
            }

print(f'GeoNames LOC lookup entries: {len(geonames_loc_lookup)}')

# Check overlap with Adkins LOC entities
adkins_loc = [(s, t) for s, t in entity_list if t == 'LOC']
hits    = [(s, t) for s, t in adkins_loc if s in geonames_loc_lookup]
misses  = [(s, t) for s, t in adkins_loc if s not in geonames_loc_lookup]

print(f'\nAdkins LOC entities:    {len(adkins_loc)}')
print(f'GeoNames hits:          {len(hits)} ({len(hits)/len(adkins_loc):.1%})')
print(f'GeoNames misses:        {len(misses)}')
print(f'\nSample hits:')
for s, t in hits[:10]:
    entry = geonames_loc_lookup[s]
    print(f'  {s} -> {entry["english_name"]} [{entry["feature_code"]}] ({entry["lang"]})')
print(f'\nSample misses:')
for s, t in misses[:10]:
    print(f'  {s}')

GeoNames LOC lookup entries: 27866

Adkins LOC entities:    558
GeoNames hits:          51 (9.1%)
GeoNames misses:        507

Sample hits:
  Arlington -> Arlington [HTL] (en)
  Baile Shláine -> Slane [PPL] (ga)
  Baile an Mhóta -> Ballymount [PPL] (ga)
  Baile na Cille -> Ballynakill [PPL] (ga)
  Baile na nGall -> Ballynagaul [PPL] (ga)
  Baile Átha Cliath -> Dublin City [ADM2] (ga)
  Baile Átha Luain -> Athlone [PPL] (ga)
  Bealach an Doirín -> Ballaghaderreen [PPL] (ga)
  Bá Bheanntraí -> Bantry Bay [BAY] (ga)
  Béal Feirste -> Belfarsad [LCTY] (ga)

Sample misses:
  14 Sráid Henrietta
  Aberdeen
  Abhainn Mhór
  Achadh Mhór
  Aerfort Haneda
  Aerfort Idirnáisiúnta Thóiceo
  Aeriris
  Afraic Theas
  Amharclann Choláiste Mhuire , Baile Átha cliath
  An Bainisteoir , Ionad an Bhlascaoid Mhóir , Dún Chaoin , Trá Lí , Co. Chiarraí


In [10]:
# CELL 9 — GeoNames LOC overlap: exact match + token-level fallback

def extract_place_tokens(surface: str) -> list[str]:
    """
    Split a compound LOC surface into candidate place-name tokens.
    Strips address numbers, punctuation clusters, and short function words.
    """
    STOPWORDS = {
        'an', 'na', 'a', 'i', 'de', 'do', 'le', 'ag', 'ar', 'as',
        'sa', 'den', 'don', 'leis', 'faoi', 'ionad', 'sráid', 'bóthar',
        'ascaill', 'plás', 'cé', 'cearnóg', 'lána', 'teach', 'halla',
        'amharclann', 'aerfort', 'ollscoil', 'coláiste', 'institiúid',
    }
    # Remove leading digits (address numbers)
    surface = re.sub(r'^\d+\s*', '', surface)
    # Split on whitespace and commas
    tokens = re.split(r'[\s,]+', surface.strip())
    # Filter: length >= 4, not a stopword, not purely numeric
    tokens = [
        t for t in tokens
        if len(t) >= 4 and t.lower() not in STOPWORDS and not t.isdigit()
    ]
    return tokens

# Exact match
exact_hits   = {s for s, t in adkins_loc if s in geonames_loc_lookup}
exact_misses = [(s, t) for s, t in adkins_loc if s not in geonames_loc_lookup]

# Token-level fallback on misses
token_hits   = {}   # surface -> list of matched tokens
token_misses = []

for surface, typ in exact_misses:
    tokens  = extract_place_tokens(surface)
    matched = [tok for tok in tokens if tok in geonames_loc_lookup]
    if matched:
        token_hits[surface] = matched
    else:
        token_misses.append(surface)

total_covered = len(exact_hits) + len(token_hits)
print(f'Exact matches:              {len(exact_hits)} ({len(exact_hits)/len(adkins_loc):.1%})')
print(f'Token-level additional:     {len(token_hits)} ({len(token_hits)/len(adkins_loc):.1%})')
print(f'Total covered:              {total_covered} ({total_covered/len(adkins_loc):.1%})')
print(f'Still unresolved:           {len(token_misses)}')

print(f'\nSample token hits:')
for s, toks in list(token_hits.items())[:10]:
    print(f'  "{s}"')
    for tok in toks:
        entry = geonames_loc_lookup[tok]
        print(f'    -> {tok} => {entry["english_name"]} [{entry["feature_code"]}]')

print(f'\nSample still-unresolved:')
for s in token_misses[:15]:
    print(f'  {s}')

Exact matches:              51 (9.1%)
Token-level additional:     17 (3.0%)
Total covered:              68 (12.2%)
Still unresolved:           490

Sample token hits:
  "Cluain Eochaille"
    -> Cluain => Cloyne [PPL]
  "Dr Douglas Hyde Centre"
    -> Douglas => Douglas [PPL]
  "Ghaeltacht Conamara"
    -> Conamara => Connemara [AREA]
  "Inis Mór"
    -> Inis => Ennis [PPLA2]
  "Leabharlann Mheal Charleville"
    -> Charleville => Charleville [EST]
  "Leabharlann Phoiblí Mheal Charleville"
    -> Charleville => Charleville [EST]
  "Liatroim Thuaidh"
    -> Liatroim => Leitrim [PPL]
  "Mainistir Naomh Tomás"
    -> Mainistir => Mainistir [PPL]
  "Moore Street"
    -> Moore => Moore [PPL]
    -> Street => Street [PPL]
  "Móinín na gCiseach"
    -> Móinín => Moneen [LCTY]

Sample still-unresolved:
  14 Sráid Henrietta
  Aberdeen
  Abhainn Mhór
  Achadh Mhór
  Aerfort Haneda
  Aerfort Idirnáisiúnta Thóiceo
  Aeriris
  Afraic Theas
  Amharclann Choláiste Mhuire , Baile Átha cliath
  An Bain

In [11]:
# CELL 10 — Irish Wikipedia title dump as primary Phase C entity source

import gzip
import urllib.request

WIKI_DUMP_URL   = 'https://dumps.wikimedia.org/gawiki/latest/gawiki-latest-all-titles-in-ns0.gz'
WIKI_DUMP_LOCAL = '/kaggle/working/gawiki-titles.gz'

if not os.path.exists(WIKI_DUMP_LOCAL):
    print('Downloading Irish Wikipedia title dump...')
    urllib.request.urlretrieve(WIKI_DUMP_URL, WIKI_DUMP_LOCAL)
    print('Done.')
else:
    print('Using cached dump.')

with gzip.open(WIKI_DUMP_LOCAL, 'rt', encoding='utf-8') as f:
    wiki_titles = [line.strip().replace('_', ' ') for line in f if line.strip()]

print(f'Irish Wikipedia titles: {len(wiki_titles)}')

# Build both exact and case-folded lookups
# case-folded: lowercase -> canonical title
wiki_title_set    = set(wiki_titles)
wiki_lower_lookup = {t.lower(): t for t in wiki_titles}  # last-write wins on collision

def wiki_lookup(surface: str) -> str | None:
    """Return canonical Wikipedia title for surface, or None. Exact first, then case-fold."""
    if surface in wiki_title_set:
        return surface
    return wiki_lower_lookup.get(surface.lower())

# Coverage check
hits_exact     = []
hits_casefold  = []
misses         = []

for s, t in entity_list:
    if s in wiki_title_set:
        hits_exact.append((s, t))
    elif wiki_lower_lookup.get(s.lower()):
        hits_casefold.append((s, t))
    else:
        misses.append((s, t))

total_hits = len(hits_exact) + len(hits_casefold)
print(f'\nAdkins entities total:    {len(entity_list)}')
print(f'Exact hits:               {len(hits_exact)} ({len(hits_exact)/len(entity_list):.1%})')
print(f'Case-fold additional:     {len(hits_casefold)} ({len(hits_casefold)/len(entity_list):.1%})')
print(f'Total covered:            {total_hits} ({total_hits/len(entity_list):.1%})')

for typ in ['PER', 'ORG', 'LOC']:
    type_total     = [(s, t) for s, t in entity_list      if t == typ]
    type_exact     = [(s, t) for s, t in hits_exact       if t == typ]
    type_casefold  = [(s, t) for s, t in hits_casefold    if t == typ]
    type_covered   = len(type_exact) + len(type_casefold)
    print(f'  {typ}: {type_covered}/{len(type_total)} ({type_covered/len(type_total):.1%})'
          f'  [exact={len(type_exact)}, casefold={len(type_casefold)}]')

print(f'\nSample case-fold hits:')
for s, t in hits_casefold[:10]:
    canonical = wiki_lower_lookup[s.lower()]
    print(f'  [{t}] "{s}" -> "{canonical}"')

print(f'\nSample still-unresolved:')
for s, t in misses[:15]:
    print(f'  [{t}] {s}')

Done.
Irish Wikipedia titles: 83429

Adkins entities total:    1863
Exact hits:               426 (22.9%)
Case-fold additional:     19 (1.0%)
Total covered:            445 (23.9%)
  PER: 154/651 (23.7%)  [exact=151, casefold=3]
  ORG: 118/654 (18.0%)  [exact=111, casefold=7]
  LOC: 173/558 (31.0%)  [exact=164, casefold=9]

Sample case-fold hits:
  [ORG] "AN tAONTAS EORPACH" -> "An tAontas Eorpach"
  [ORG] "ANOIS" -> "Anois"
  [LOC] "BHAILE ÁTHA CLIATH" -> "Bhaile Átha Cliath"
  [LOC] "CHONAMARA" -> "Chonamara"
  [ORG] "COMHAIRLE CHONTAE NA GAILLIMHE" -> "Comhairle Chontae na Gaillimhe"
  [ORG] "DEIS" -> "Deis"
  [ORG] "Iontaobhas Ultach" -> "Iontaobhas ULTACH"
  [ORG] "Lucht Oibre" -> "Lucht oibre"
  [LOC] "Oifig an Phoist" -> "Oifig an phoist"
  [ORG] "PAS" -> "Pas"

Sample still-unresolved:
  [ORG] ( Y ) OUR STORY
  [ORG] AIE
  [ORG] ATU
  [PER] Aaron O’Shea
  [LOC] Aberdeen
  [ORG] Acadamh an Bhaile Meánach
  [LOC] Achadh Mhór
  [LOC] Aerfort Haneda
  [LOC] Aerfort Idirnáisiúnta Thó

In [12]:
# CELL 11 — systematic miss analysis

from collections import Counter
import re

misses_by_type = {'PER': [], 'ORG': [], 'LOC': []}
for s, t in misses:
    misses_by_type[t].append(s)

def classify_miss(surface: str, typ: str) -> str:
    # Address / compound with comma
    if ',' in surface:
        return 'compound_address'
    # Leading digit
    if re.match(r'^\d', surface):
        return 'address_number'
    # All caps, length <= 5
    if surface.isupper() and len(surface) <= 5:
        return 'acronym'
    # All caps, longer
    if surface.isupper():
        return 'allcaps'
    # Contains a space and looks like "Firstname Surname" (PER)
    if typ == 'PER' and len(surface.split()) == 2:
        return 'person_name'
    if typ == 'PER' and len(surface.split()) >= 3:
        return 'person_name_long'
    # Looks like a place (LOC) — Irish morphology markers
    PLACE_PREFIXES = ['An ', 'Na ', 'Baile', 'Loch', 'Béal', 'Bá ',
                      'Abhainn', 'Cnoc', 'Inis', 'Cill', 'Dún', 'Ros',
                      'Droichead', 'Gleann', 'Carraig', 'Ath ', 'Áth ']
    if typ == 'LOC' and any(surface.startswith(p) for p in PLACE_PREFIXES):
        return 'irish_placename'
    # Foreign place
    if typ == 'LOC':
        return 'loc_other'
    return 'other'

miss_categories = Counter()
classified = {}
for typ, surfaces in misses_by_type.items():
    for s in surfaces:
        cat = classify_miss(s, typ)
        miss_categories[cat] += 1
        classified[s] = (typ, cat)

print('Miss classification:')
for cat, count in miss_categories.most_common():
    print(f'  {cat:<25} {count}')

print(f'\nTotal misses: {sum(miss_categories.values())}')

# Show full list per category so we can assess recoverability
for cat in ['person_name', 'irish_placename', 'allcaps', 'loc_other']:
    examples = [s for s, (t, c) in classified.items() if c == cat]
    print(f'\n--- {cat} ({len(examples)}) ---')
    for s in examples[:30]:
        print(f'  {s}')

Miss classification:
  other                     605
  loc_other                 349
  person_name               214
  person_name_long          141
  acronym                   49
  compound_address          34
  irish_placename           22
  allcaps                   4

Total misses: 1418

--- person_name (214) ---
  Aaron O’Shea
  Agent Orange
  Aiden Coffey
  Aire Airgeadais
  Aire Iompair
  Aire Oideachais
  Aire Sláinte
  Aire Stáit
  Airí Stáit
  Akhil Sharma
  An Pionta
  An Seanfhear
  An Strainséara
  Antain Mór
  Aodh Rua
  Ard Mhaistéara
  Barry Keoghan
  Bertti Vogts
  Beverley Cooper-Flynn
  Bhanríon Shasana
  Bob Collins
  Bob Kansas
  Brendan McHale
  Brendan Teeling
  Briain Cairt
  Brian McEniff
  Bronagh O’Hanlon
  C.J. Dolan
  Cathaoirleach Gníomhach
  Celia Larkin

--- irish_placename (22) ---
  An Blascaod
  An Lios
  An Teampall
  Baile Bhiocáire
  Baile Bhoithín
  Baile Meánach
  Baile an Tóchair Thiar
  Baile an Tóchair Thoir
  Baile na Sí
  Carraig Beannchair


In [13]:
# CELL 12 — deep audit of 'other' and 'loc_other' buckets

# 'other' breakdown — what is actually in there?
other_surfaces = [(s, t) for s, (t, c) in classified.items() if c == 'other']

print(f'=== OTHER ({len(other_surfaces)}) — type breakdown ===')
other_by_type = Counter(t for s, t in other_surfaces)
print(other_by_type)

print(f'\n--- OTHER: PER samples ---')
for s, t in [(s, t) for s, t in other_surfaces if t == 'PER'][:30]:
    print(f'  {s}')

print(f'\n--- OTHER: ORG samples ---')
for s, t in [(s, t) for s, t in other_surfaces if t == 'ORG'][:30]:
    print(f'  {s}')

print(f'\n--- OTHER: LOC samples ---')
for s, t in [(s, t) for s, t in other_surfaces if t == 'LOC'][:30]:
    print(f'  {s}')

# loc_other — how many are lenited/eclipsed forms of known places?
print(f'\n\n=== LOC_OTHER ({len([s for s, (t,c) in classified.items() if c=="loc_other"])}) ===')

IRISH_MUTATIONS = {
    # Lenition prefixes
    'Bh': 'B', 'Ch': 'C', 'Dh': 'D', 'Fh': 'F', 'Gh': 'G',
    'Mh': 'M', 'Ph': 'P', 'Sh': 'S', 'Th': 'T',
    # Eclipsis prefixes  
    'mb': 'B', 'gc': 'C', 'nd': 'D', 'bhf': 'F', 'ng': 'G',
    'bp': 'P', 'dt': 'T',
    # t- and h- prefixes
    't-': '', 'h': '',
    # Capitalised eclipsis
    'Mb': 'B', 'Gc': 'C', 'Nd': 'D', 'Bhf': 'F', 'Ng': 'G',
    'Bp': 'P', 'Dt': 'T',
}

def demutate(surface: str) -> list[str]:
    """Generate candidate base forms by stripping Irish initial mutations."""
    candidates = [surface]
    for prefix, replacement in IRISH_MUTATIONS.items():
        if surface.startswith(prefix):
            stem = surface[len(prefix):]
            if replacement:
                candidates.append(replacement + stem)
                candidates.append(replacement.lower() + stem)
            else:
                candidates.append(stem)
                if len(stem) > 0:
                    candidates.append(stem[0].upper() + stem[1:])
    return list(dict.fromkeys(candidates))  # deduplicate, preserve order

loc_other_surfaces = [s for s, (t, c) in classified.items() if c == 'loc_other']

demutation_hits  = {}
demutation_still_miss = []

for surface in loc_other_surfaces:
    candidates = demutate(surface)
    hit = None
    for cand in candidates:
        if cand in wiki_title_set:
            hit = cand
            break
        if cand.lower() in wiki_lower_lookup:
            hit = wiki_lower_lookup[cand.lower()]
            break
    if hit:
        demutation_hits[surface] = hit
    else:
        demutation_still_miss.append(surface)

print(f'LOC demutation hits:    {len(demutation_hits)} ({len(demutation_hits)/len(loc_other_surfaces):.1%})')
print(f'LOC still unresolved:   {len(demutation_still_miss)}')

print(f'\nSample demutation hits:')
for s, hit in list(demutation_hits.items())[:20]:
    print(f'  {s} -> {hit}')

print(f'\nSample still unresolved after demutation:')
for s in demutation_still_miss[:20]:
    print(f'  {s}')

=== OTHER (605) — type breakdown ===
Counter({'ORG': 469, 'PER': 136})

--- OTHER: PER samples ---
  Aire
  Ali
  Andrews
  Anne-Marie
  Anthony
  Ard-Reachtaire
  Barranikov
  Berry
  Bhanríon
  Bhreandáin
  Bhreandán
  Bhriain
  Brahe
  Breandán
  Brendan
  Bríde
  Burnside
  Butler
  Caidé
  Caimbeul
  Caitríona
  Calpruinn
  Charleton
  Choilm
  Chonaire
  Chormaic
  Chrócaigh
  Churchill
  Cháit
  Ciarán

--- OTHER: ORG samples ---
  Acadamh an Bhaile Meánach
  Aga Khan
  Aisling na nÓg
  Amharclann de hÍde
  An Chomhairle Ealaíon
  An Páirtí Glas
  An Roinn Ceapachán agus Forbairt Foirne
  An Roinn Gnóthaí Pobail Tuaithe agus Gaeltachta
  An Roinn Nuálaíochta agus Teicneolaíochta
  An Roinn Oideachais agus Eolaíochta
  Aonad Agrai-Éiceolaíochta
  Aontachtóir Ultach
  Aontais
  Aontais ballstát
  Aontas
  Arm Cathartha
  Arm Slánaithe
  Bailiúchán Éireann
  Banna Ceoil Schomberg
  Barcó
  Barnardos
  Belfast Regeneration Office
  Belfast Society for Promoting Knowledge
  Belfast T

In [14]:
# CELL 13 — demutation pass on ORG misses + Wikidata SPARQL for PER

# --- Part A: demutation on ORG misses ---

org_other_surfaces = [s for s, (t, c) in classified.items()
                      if c == 'other' and t == 'ORG']

org_demutation_hits  = {}
org_demutation_miss  = []

for surface in org_other_surfaces:
    candidates = demutate(surface)
    hit = None
    for cand in candidates:
        if cand in wiki_title_set:
            hit = cand
            break
        if cand.lower() in wiki_lower_lookup:
            hit = wiki_lower_lookup[cand.lower()]
            break
    if hit:
        org_demutation_hits[surface] = hit
    else:
        org_demutation_miss.append(surface)

print(f'ORG demutation hits:  {len(org_demutation_hits)} / {len(org_other_surfaces)} '
      f'({len(org_demutation_hits)/len(org_other_surfaces):.1%})')

print(f'\nSample ORG demutation hits:')
for s, hit in list(org_demutation_hits.items())[:20]:
    print(f'  {s} -> {hit}')

print(f'\nSample ORG still unresolved:')
for s in org_demutation_miss[:20]:
    print(f'  {s}')

# --- Part B: demutation on PER 'other' (single tokens — audit only, not for KG) ---

per_other_surfaces = [s for s, (t, c) in classified.items()
                      if c == 'other' and t == 'PER']

per_demutation_hits = {}
for surface in per_other_surfaces:
    candidates = demutate(surface)
    for cand in candidates:
        if cand in wiki_title_set:
            per_demutation_hits[surface] = cand
            break
        if cand.lower() in wiki_lower_lookup:
            per_demutation_hits[surface] = wiki_lower_lookup[cand.lower()]
            break

print(f'\n\nPER single-token demutation hits: {len(per_demutation_hits)} / {len(per_other_surfaces)}')
print('(audit only — single-token mentions are partial; inspect before including)')
print(f'\nSample PER single-token hits:')
for s, hit in list(per_demutation_hits.items())[:20]:
    print(f'  {s} -> {hit}')

# --- Part C: running coverage tally ---

newly_covered = (
    len(org_demutation_hits)
    + len(demutation_hits)       # LOC from Cell 12
)

prev_covered  = 445
total_covered_now = prev_covered + newly_covered

print(f'\n\n=== Coverage tally ===')
print(f'After Wikipedia exact + casefold:  {prev_covered} / 1863 ({prev_covered/1863:.1%})')
print(f'LOC demutation (Cell 12):          +{len(demutation_hits)}')
print(f'ORG demutation (Cell 13):          +{len(org_demutation_hits)}')
print(f'Running total:                     {total_covered_now} / 1863 ({total_covered_now/1863:.1%})')
print(f'\nStill unresolved: {1863 - total_covered_now}')
print(f'  PER person_name (Wikidata target): 214')
print(f'  PER single-token partial:          {len(per_other_surfaces) - len(per_demutation_hits)}')
print(f'  ORG still miss:                    {len(org_demutation_miss)}')
print(f'  LOC foreign (redirect target):     ~299')

ORG demutation hits:  21 / 469 (4.5%)

Sample ORG demutation hits:
  Chathair Chorcaí -> Cathair Chorcaí
  Chlann na Poblachta -> Clann na Poblachta
  Chomhairle Cathrach Bhaile Átha Cliath -> Comhairle Cathrach Bhaile Átha Cliath
  Chomhairle Chontae na Gaillimhe -> Comhairle Chontae na Gaillimhe
  Chomhairle Contae -> Comhairle Contae
  Chomhairle Contae na Gaillimhe -> Comhairle Contae na Gaillimhe
  Chonradh -> Conradh
  Chonradh na Gaeilge -> Conradh na Gaeilge
  Dháil Éireann -> Dáil Éireann
  Fhianna Fáil -> Fianna Fáil
  Fhoras na Gaeilge -> Foras na Gaeilge
  Ghael-linn -> Gael-Linn
  Gharda Síochána -> Garda Síochána
  Mhacra na Feirme -> Macra na Feirme
  Mhuintearas -> Muintearas
  Pháirtí na nOibrithe -> Páirtí na nOibrithe
  Thithe an Oireachtais -> Tithe an Oireachtais
  hAcadamh na hOllscolaíochta Gaeilge -> Acadamh na hOllscolaíochta Gaeilge
  hAlban -> Alban
  hÉireannaigh Aontaithe -> Éireannaigh Aontaithe

Sample ORG still unresolved:
  Acadamh an Bhaile Meánach
  A

In [15]:
# CELL 14 — Wikidata label matching for unresolved PER, ORG, LOC

import requests
import time
from collections import defaultdict

WIKIDATA_SPARQL = 'https://query.wikidata.org/sparql'

def query_wikidata_labels(surfaces: list[str], lang: str = 'ga') -> dict[str, dict]:
    """
    Batch lookup: for each surface string, find Wikidata items whose
    Irish-language label or alias exactly matches.
    Returns dict: surface -> {'qid', 'label', 'description', 'instance_of'}
    """
    # Wikidata VALUES clause — batch in groups of 50 to avoid timeout
    results = {}
    
    def run_batch(batch: list[str]) -> None:
        values = ' '.join(f'"{s}"@{lang}' for s in batch)
        query = f"""
SELECT DISTINCT ?item ?itemLabel ?itemDescription ?instance_ofLabel WHERE {{
  VALUES ?searchLabel {{ {values} }}
  ?item rdfs:label|skos:altLabel ?searchLabel .
  OPTIONAL {{ ?item wdt:P31 ?instance_of . }}
  SERVICE wikibase:label {{
    bd:serviceParam wikibase:language "{lang},en" .
  }}
}}
LIMIT 200
"""
        try:
            resp = requests.get(
                WIKIDATA_SPARQL,
                params={'query': query, 'format': 'json'},
                headers={'User-Agent': 'IrishNER-dissertation/1.0'},
                timeout=30
            )
            if resp.status_code != 200:
                return
            bindings = resp.json()['results']['bindings']
            for b in bindings:
                label = b.get('itemLabel', {}).get('value', '')
                if label in batch:
                    qid  = b['item']['value'].split('/')[-1]
                    desc = b.get('itemDescription', {}).get('value', '')
                    inst = b.get('instance_ofLabel', {}).get('value', '')
                    if label not in results:
                        results[label] = {
                            'qid':         qid,
                            'label':       label,
                            'description': desc,
                            'instance_of': inst,
                        }
        except Exception as e:
            print(f'  SPARQL error: {e}')

    # Build target list — exclude single-token PER partials
    batch_size = 50
    for i in range(0, len(surfaces), batch_size):
        batch = surfaces[i:i + batch_size]
        run_batch(batch)
        print(f'  {min(i+batch_size, len(surfaces))}/{len(surfaces)} queried, '
              f'{len(results)} hits so far')
        time.sleep(1.0)

    return results

# Targets
per_targets = [s for s, (t, c) in classified.items()
               if t == 'PER' and c == 'person_name']
org_targets = [s for s in org_demutation_miss]
loc_targets = [s for s in demutation_still_miss]

print(f'Wikidata targets:')
print(f'  PER: {len(per_targets)}')
print(f'  ORG: {len(org_targets)}')
print(f'  LOC: {len(loc_targets)}')

all_targets = list(dict.fromkeys(per_targets + org_targets + loc_targets))
print(f'  Total unique: {len(all_targets)}')
print(f'\nQuerying Wikidata...')

wikidata_hits = query_wikidata_labels(all_targets, lang='ga')

# Separate back by type
per_wd_hits = {s: wikidata_hits[s] for s in per_targets if s in wikidata_hits}
org_wd_hits = {s: wikidata_hits[s] for s in org_targets if s in wikidata_hits}
loc_wd_hits = {s: wikidata_hits[s] for s in loc_targets if s in wikidata_hits}

print(f'\n=== Wikidata results ===')
print(f'PER hits: {len(per_wd_hits)} / {len(per_targets)} ({len(per_wd_hits)/len(per_targets):.1%})')
print(f'ORG hits: {len(org_wd_hits)} / {len(org_targets)} ({len(org_wd_hits)/len(org_targets):.1%})')
print(f'LOC hits: {len(loc_wd_hits)} / {len(loc_targets)} ({len(loc_wd_hits)/len(loc_targets):.1%})')

total_wd = len(per_wd_hits) + len(org_wd_hits) + len(loc_wd_hits)
grand_total = 516 + total_wd
print(f'\nWikidata addition:   +{total_wd}')
print(f'Grand total covered: {grand_total} / 1863 ({grand_total/1863:.1%})')

print(f'\nSample PER hits:')
for s, r in list(per_wd_hits.items())[:10]:
    print(f'  {s} -> {r["qid"]} | {r["description"][:60]}')

print(f'\nSample ORG hits:')
for s, r in list(org_wd_hits.items())[:10]:
    print(f'  {s} -> {r["qid"]} | {r["description"][:60]}')

print(f'\nSample LOC hits:')
for s, r in list(loc_wd_hits.items())[:10]:
    print(f'  {s} -> {r["qid"]} | {r["description"][:60]}')

Wikidata targets:
  PER: 214
  ORG: 448
  LOC: 299
  Total unique: 961

Querying Wikidata...
  50/961 queried, 8 hits so far
  100/961 queried, 21 hits so far
  150/961 queried, 33 hits so far
  200/961 queried, 42 hits so far
  250/961 queried, 51 hits so far
  300/961 queried, 52 hits so far
  350/961 queried, 54 hits so far
  400/961 queried, 58 hits so far
  450/961 queried, 61 hits so far
  500/961 queried, 66 hits so far
  550/961 queried, 70 hits so far
  600/961 queried, 73 hits so far
  650/961 queried, 73 hits so far
  700/961 queried, 80 hits so far
  750/961 queried, 85 hits so far
  800/961 queried, 87 hits so far
  850/961 queried, 96 hits so far
  900/961 queried, 96 hits so far
  950/961 queried, 96 hits so far
  961/961 queried, 96 hits so far

=== Wikidata results ===
PER hits: 47 / 214 (22.0%)
ORG hits: 28 / 448 (6.2%)
LOC hits: 21 / 299 (7.0%)

Wikidata addition:   +96
Grand total covered: 612 / 1863 (32.9%)

Sample PER hits:
  Agent Orange -> Q392711 | 1989 studio 

In [16]:
# CELL 15 — Wikidata hit quality audit and filtering

IRELAND_SIGNALS = {
    'Éireann', 'Éire', 'Éireannach', 'Irish', 'Ireland',
    'Northern Ireland', 'Thuaisceart Éireann', 'Baile Átha Cliath',
    'Dublin', 'Cork', 'Galway', 'Belfast',
}

PERSON_SIGNALS = {
    'aisteoir', 'iriseoir', 'scríbhneoir', 'polaiteoir', 'amhránaí',
    'saoránach', 'státseirbhíseach', 'múinteoir', 'dochtúir',
    'actor', 'politician', 'journalist', 'writer', 'singer',
    'footballer', 'player', 'director', 'author', 'poet',
    'minister', 'senator', 'deputy', 'teachta',
}

ORG_SIGNALS = {
    'agency', 'organisation', 'organization', 'company', 'institution',
    'association', 'foundation', 'society', 'council', 'committee',
    'department', 'university', 'college', 'school', 'comhairle',
    'roinn', 'institiúid', 'cumann', 'coláiste', 'eagraíocht',
}

LOC_SIGNALS = {
    'townland', 'town', 'city', 'county', 'village', 'island',
    'river', 'lake', 'mountain', 'region', 'country', 'province',
    'toghroinn', 'contae', 'baile', 'loch', 'oileán', 'abhainn',
}

DISCARD_SIGNALS = {
    'disambiguation', 'díchomhthéacs', 'Wikimedia',
    'surname', 'sloinne', 'given name', 'ainm baiste',
    'album', 'film', 'scannán', 'song', 'amhrán',
    'television', 'video game', 'manga', 'anime',
}

def score_hit(surface: str, entity_type: str, hit: dict) -> tuple[str, str]:
    """
    Returns (verdict, reason): 'keep', 'review', or 'discard'
    """
    desc = (hit.get('description') or '').lower()
    inst = (hit.get('instance_of') or '').lower()
    combined = desc + ' ' + inst

    # Hard discard
    for sig in DISCARD_SIGNALS:
        if sig.lower() in combined:
            return 'discard', f'discard signal: "{sig}"'

    # Type-specific checks
    ireland_match = any(s.lower() in combined for s in IRELAND_SIGNALS)

    if entity_type == 'PER':
        person_match = any(s.lower() in combined for s in PERSON_SIGNALS)
        if person_match and ireland_match:
            return 'keep', 'Irish person confirmed'
        if person_match:
            return 'review', 'person but not confirmed Irish'
        return 'discard', 'no person signal in description'

    if entity_type == 'ORG':
        org_match = any(s.lower() in combined for s in ORG_SIGNALS)
        if org_match and ireland_match:
            return 'keep', 'Irish organisation confirmed'
        if org_match:
            return 'review', 'organisation but not confirmed Irish'
        return 'discard', 'no organisation signal'

    if entity_type == 'LOC':
        loc_match = any(s.lower() in combined for s in LOC_SIGNALS)
        if loc_match:
            return 'keep', 'location signal present'
        return 'discard', 'no location signal'

    return 'review', 'unclassified'

# Apply to all Wikidata hits
verdicts = {'keep': [], 'review': [], 'discard': []}

for typ, hits_dict in [('PER', per_wd_hits), ('ORG', org_wd_hits), ('LOC', loc_wd_hits)]:
    for surface, hit in hits_dict.items():
        verdict, reason = score_hit(surface, typ, hit)
        verdicts[verdict].append((surface, typ, hit, reason))

print(f'=== Wikidata hit quality ===')
for v in ['keep', 'review', 'discard']:
    print(f'  {v}: {len(verdicts[v])}')

print(f'\n--- KEEP ({len(verdicts["keep"])}) ---')
for s, t, h, r in verdicts['keep']:
    print(f'  [{t}] {s} -> {h["qid"]} | {h["description"][:70]}')

print(f'\n--- REVIEW ({len(verdicts["review"])}) ---')
for s, t, h, r in verdicts['review']:
    print(f'  [{t}] {s} -> {h["qid"]} | {h["description"][:70]} | {r}')

print(f'\n--- DISCARD ({len(verdicts["discard"])}) ---')
for s, t, h, r in verdicts['discard']:
    print(f'  [{t}] {s} -> {h["qid"]} | {h["description"][:70]} | {r}')

=== Wikidata hit quality ===
  keep: 21
  review: 13
  discard: 62

--- KEEP (21) ---
  [PER] Barry Keoghan -> Q28542230 | aisteoir Éireannach
  [PER] Celia Larkin -> Q5058005 | státseirbhíseach Éireannach
  [PER] Eoin McNamee -> Q5381731 | scríbhneoir Éireannach
  [PER] Oliver Goldsmith -> Q236236 | scríbhneoir agus dochtúir Éireannach
  [ORG] An Chomhairle Ealaíon -> Q4801451 | Irish government agency
  [ORG] Coláiste Ghobnatan -> Q5141882 | school in Republic of Ireland
  [ORG] Cultúr Éireann -> Q1672824 | Irish State cultural agency
  [ORG] Údarás Forbartha Dugthailte Bhaile Átha Cliath -> Q5310880 | Irish government agency
  [LOC] Achadh Mhór -> Q104328030 | townland in Carrigallen East, County Leitrim, Ireland
  [LOC] Arlington -> Q947945 | village in Gloucestershire, England
  [LOC] Cluain Eochaille -> Q59731513 | toghroinn i gContae Shligigh
  [LOC] Droim Fionn -> Q59719046 | toghroinn i gContae Bhaile Átha Cliath
  [LOC] Liatroim Thuaidh -> Q104340906 | townland in Cloonacool,

In [17]:
# CELL 16 — corrected scoring with expanded Irish signals + manual overrides

IRELAND_SIGNALS = {
    # English
    'irish', 'ireland', 'republic of ireland', 'northern ireland',
    'dublin', 'cork', 'galway', 'belfast', 'limerick', 'waterford',
    # Irish
    'éireann', 'éire', 'éireannach', 'na héireann', 'poblacht na héireann',
    'baile átha cliath', 'thuaisceart éireann',
}

PERSON_SIGNALS = {
    # English roles/professions
    'actor', 'actress', 'politician', 'journalist', 'writer', 'singer',
    'footballer', 'player', 'director', 'author', 'poet', 'minister',
    'senator', 'deputy', 'artist', 'musician', 'composer', 'priest',
    'bishop', 'judge', 'lawyer', 'doctor', 'scientist', 'academic',
    'businessman', 'businesswoman', 'activist', 'soldier', 'officer',
    'broadcaster', 'presenter', 'comedian', 'novelist', 'historian',
    'philosopher', 'theologian', 'archbishop', 'cardinal', 'rugby',
    'cricketer', 'swimmer', 'athlete', 'boxer', 'cyclist', 'rower',
    # Irish professions/descriptors
    'aisteoir', 'iriseoir', 'scríbhneoir', 'polaiteoir', 'amhránaí',
    'státseirbhíseach', 'múinteoir', 'dochtúir', 'ealaíontóir',
    'ceoltóir', 'cumadóir', 'sagart', 'easpag', 'dlíodóir',
    'fear gnó', 'bean ghnó', 'gníomhaí', 'saoránach', 'peileadóir',
    'iomróir', 'lúthchleasaí', 'snámhóir', 'rothlaí', 'dornálaí',
    'craoltóir', 'údar', 'file', 'staraí', 'fealsúnaí', 'diagaire',
    # Born/died patterns
    'born', 'died', '(1', '(2',
}

ORG_SIGNALS = {
    # English
    'agency', 'organisation', 'organization', 'company', 'institution',
    'association', 'foundation', 'society', 'council', 'committee',
    'department', 'university', 'college', 'school', 'board', 'office',
    'authority', 'body', 'commission', 'trust', 'charity', 'union',
    'newspaper', 'broadcaster', 'publisher', 'gallery', 'museum',
    'library', 'theatre', 'hospital', 'bank', 'party', 'movement',
    # Irish
    'comhairle', 'roinn', 'institiúid', 'cumann', 'coláiste',
    'eagraíocht', 'údarás', 'bord', 'oifig', 'coimisiún',
    'ollscoil', 'scoil', 'ospidéal', 'banc', 'páirtí', 'gluaiseacht',
    'dánlann', 'músaem', 'leabharlann', 'amharclann', 'nuachtán',
}

LOC_SIGNALS = {
    # English
    'townland', 'town', 'city', 'county', 'village', 'island',
    'river', 'lake', 'mountain', 'region', 'country', 'province',
    'district', 'area', 'bay', 'peninsula', 'valley', 'parish',
    'electoral', 'settlement', 'municipality', 'commune', 'state',
    'republic', 'kingdom', 'empire', 'territory', 'continent',
    'ocean', 'sea', 'strait', 'cape', 'coast', 'harbour', 'port',
    'unincorporated community', 'census-designated',
    # Irish
    'toghroinn', 'contae', 'baile', 'loch', 'oileán', 'abhainn',
    'paróiste', 'cathair', 'sráidbhaile', 'ceantar', 'réigiún',
    'poblacht', 'ríocht', 'críoch', 'mór-roinn', 'farraige',
}

DISCARD_SIGNALS = {
    'disambiguation', 'díchomhthéacs', 'wikimedia',
    'surname', 'sloinne', 'given name', 'ainm baiste', 'forainm',
    'album', 'film', 'song', 'amhrán', 'video game', 'manga', 'anime',
    'television series', 'sraith theilifíse', 'sraith teilifíse',
    'comic', 'fictional',
}

# Manual overrides: surface -> 'keep' or 'discard'
MANUAL_OVERRIDES = {
    'Leas-Cheann Comhairle':              'keep',
    'Oifig Aicmithe Scannán na hÉireann': 'keep',
    'Winnie Ewing':                       'keep',
    'Stephen O\'Brien':                   'discard',  # British politician, wrong referent
    'Bob Collins':                        'discard',  # Australian footballer
    'David Beresford':                    'discard',  # South African journalist
    'Kevin Kelly':                        'discard',  # American author
    'Margaret Hayes':                     'discard',  # American actress
    'Pat Harvey':                         'discard',  # American journalist
    'Paul Lambert':                       'discard',  # American politician
    'Peter Kelly':                        'discard',  # American soccer player
    'Tom Hulce':                          'discard',  # American actor
}

def score_hit_v2(surface: str, entity_type: str, hit: dict) -> tuple[str, str]:
    if surface in MANUAL_OVERRIDES:
        return MANUAL_OVERRIDES[surface], 'manual override'

    desc = (hit.get('description') or '').lower()
    inst = (hit.get('instance_of') or '').lower()
    combined = desc + ' ' + inst

    for sig in DISCARD_SIGNALS:
        if sig in combined:
            return 'discard', f'discard signal: "{sig}"'

    ireland_match = any(s in combined for s in IRELAND_SIGNALS)
    scannán_check = 'scannán' in combined  # Irish for film — catch missed cases

    if scannán_check:
        return 'discard', 'discard signal: "scannán"'

    if entity_type == 'PER':
        person_match = any(s in combined for s in PERSON_SIGNALS)
        if person_match:
            return 'keep', f'person confirmed (ireland={ireland_match})'
        return 'discard', 'no person signal'

    if entity_type == 'ORG':
        org_match = any(s in combined for s in ORG_SIGNALS)
        if org_match:
            return 'keep', f'org confirmed (ireland={ireland_match})'
        return 'discard', 'no org signal'

    if entity_type == 'LOC':
        loc_match = any(s in combined for s in LOC_SIGNALS)
        if loc_match:
            return 'keep', f'location confirmed (ireland={ireland_match})'
        return 'discard', 'no location signal'

    return 'discard', 'unclassified'

# Re-score all 92 hits
verdicts_v2 = {'keep': [], 'discard': []}

all_wd_hits = (
    [(s, 'PER', h) for s, h in per_wd_hits.items()] +
    [(s, 'ORG', h) for s, h in org_wd_hits.items()] +
    [(s, 'LOC', h) for s, h in loc_wd_hits.items()]
)

for surface, typ, hit in all_wd_hits:
    verdict, reason = score_hit_v2(surface, typ, hit)
    verdicts_v2[verdict].append((surface, typ, hit, reason))

print(f'=== Re-scored Wikidata hits ===')
print(f'  keep:    {len(verdicts_v2["keep"])}')
print(f'  discard: {len(verdicts_v2["discard"])}')

print(f'\n--- KEEP ({len(verdicts_v2["keep"])}) ---')
for s, t, h, r in verdicts_v2['keep']:
    print(f'  [{t}] {s} -> {h["qid"]} | {h["description"][:70]} | {r}')

print(f'\n--- DISCARD ({len(verdicts_v2["discard"])}) ---')
for s, t, h, r in verdicts_v2['discard']:
    print(f'  [{t}] {s} -> {h["qid"]} | {h["description"][:60]} | {r}')

# Final coverage tally
clean_wd_hits = {s for s, t, h, r in verdicts_v2['keep']}
grand_total   = 516 + len(clean_wd_hits)
print(f'\n=== Coverage tally ===')
print(f'Wikipedia + demutation:  516 / 1863 (27.7%)')
print(f'Clean Wikidata:         +{len(clean_wd_hits)}')
print(f'Grand total:             {grand_total} / 1863 ({grand_total/1863:.1%})')

=== Re-scored Wikidata hits ===
  keep:    37
  discard: 59

--- KEEP (37) ---
  [PER] Barry Keoghan -> Q28542230 | aisteoir Éireannach | person confirmed (ireland=True)
  [PER] Celia Larkin -> Q5058005 | státseirbhíseach Éireannach | person confirmed (ireland=True)
  [PER] Eoin McNamee -> Q5381731 | scríbhneoir Éireannach | person confirmed (ireland=True)
  [PER] Fabrizio Caselli -> Q102372095 | Doctoral Università di Roma La Sapienza 2002 | person confirmed (ireland=False)
  [PER] John Downey -> Q6259971 | American judge and CIA officer | person confirmed (ireland=False)
  [PER] Leas-Cheann Comhairle -> Q63706078 | Deputy Speaker of Dail Eireann | manual override
  [PER] Martin Smith -> Q16191287 | Party official, born 1963 | person confirmed (ireland=False)
  [PER] Oliver Goldsmith -> Q236236 | scríbhneoir agus dochtúir Éireannach | person confirmed (ireland=True)
  [PER] Pat McDonagh -> Q55363749 | iomróir Éireannach | person confirmed (ireland=True)
  [PER] Rupert Murdoch -> Q5394

In [18]:
# CELL 17 — final false keep removal and definitive coverage count

FALSE_KEEPS = {
    'Fabrizio Caselli',   # Roman doctoral student
    'John Downey',        # American CIA officer
    'Martin Smith',       # unverifiable party official
    'Bishopsgate Institute',  # London library
    'Arlington',          # Gloucestershire, not Irish corpus referent
    'San José',           # Cuba
    'Santa Lucia',        # Philippines
    'Garryowen',          # unincorporated US community, not Limerick
}

final_wd_keeps = [
    (s, t, h, r) for s, t, h, r in verdicts_v2['keep']
    if s not in FALSE_KEEPS
]

print(f'After false keep removal:')
print(f'  Raw keeps:     {len(verdicts_v2["keep"])}')
print(f'  False removes: {len(FALSE_KEEPS)}')
print(f'  Clean keeps:   {len(final_wd_keeps)}')

print(f'\n--- Final clean Wikidata entities ({len(final_wd_keeps)}) ---')
for s, t, h, r in final_wd_keeps:
    print(f'  [{t}] {s} -> {h["qid"]} | {h["description"][:65]}')

# Definitive coverage
clean_wd_count  = len(final_wd_keeps)
grand_total     = 516 + clean_wd_count
total_pct       = grand_total / 1863

print(f'\n=== Definitive coverage ceiling ===')
print(f'Wikipedia exact:          426  (22.9%)')
print(f'Wikipedia casefold:        19  ( 1.0%)')
print(f'LOC demutation:            50  ( 2.7%)')
print(f'ORG demutation:            21  ( 1.1%)')
print(f'Clean Wikidata:           {clean_wd_count:>3}  ({clean_wd_count/1863:.1%})')
print(f'                         ----')
print(f'Total:                    {grand_total}  ({total_pct:.1%})')
print(f'Irreducible misses:       {1863 - grand_total}  ({(1863-grand_total)/1863:.1%})')
print(f'\nIrreducible miss breakdown (approximate):')
print(f'  PER single-token partials:     ~134')
print(f'  ORG not in any KB:             ~427')
print(f'  LOC genuinely minor/foreign:   ~278')
print(f'  Other:                         ~{1863 - grand_total - 134 - 427 - 278}')

After false keep removal:
  Raw keeps:     37
  False removes: 8
  Clean keeps:   29

--- Final clean Wikidata entities (29) ---
  [PER] Barry Keoghan -> Q28542230 | aisteoir Éireannach
  [PER] Celia Larkin -> Q5058005 | státseirbhíseach Éireannach
  [PER] Eoin McNamee -> Q5381731 | scríbhneoir Éireannach
  [PER] Leas-Cheann Comhairle -> Q63706078 | Deputy Speaker of Dail Eireann
  [PER] Oliver Goldsmith -> Q236236 | scríbhneoir agus dochtúir Éireannach
  [PER] Pat McDonagh -> Q55363749 | iomróir Éireannach
  [PER] Rupert Murdoch -> Q53944 | Australian-American business magnate (born 1931)
  [PER] Winnie Ewing -> Q334015 | Scottish politician (1929–2023)
  [ORG] An Chomhairle Ealaíon -> Q4801451 | Irish government agency
  [ORG] Belfast Telegraph -> Q3985843 | British daily newspaper of Belfast, Northern Ireland
  [ORG] Coláiste Ghobnatan -> Q5141882 | school in Republic of Ireland
  [ORG] Cultúr Éireann -> Q1672824 | Irish State cultural agency
  [ORG] Dánlann Chathair Bhaile Átha Cli

In [19]:
# CELL 18 — Wikipedia redirect lookup for unresolved entities (fixed)

WIKI_API_GA = 'https://ga.wikipedia.org/w/api.php'
HEADERS = {'User-Agent': 'IrishNER-dissertation/1.0 (michael.markey@tudublin.ie)'}

def check_wiki_redirects(surfaces: list[str], batch_size: int = 50) -> dict[str, str]:
    hits = {}

    for i in range(0, len(surfaces), batch_size):
        batch = surfaces[i:i + batch_size]
        params = {
            'action':   'query',
            'titles':   '|'.join(batch),
            'redirects': '1',
            'format':   'json',
        }
        try:
            resp = requests.get(
                WIKI_API_GA,
                params=params,
                headers=HEADERS,
                timeout=15
            )
            data = resp.json()
            query = data.get('query', {})
            for redirect in query.get('redirects', []):
                source = redirect['from']
                target = redirect['to']
                if source in batch:
                    hits[source] = target

        except Exception as e:
            print(f'  Error on batch {i//batch_size}: {e}')

        time.sleep(0.2)

    return hits

already_covered = (
    {s for s, t in hits_exact}
    | {s for s, t in hits_casefold}
    | {s for s in demutation_hits}
    | {s for s in org_demutation_hits}
    | {s for s, t, h, r in final_wd_keeps}
)

all_misses = [s for s, t in entity_list if s not in already_covered]
print(f'Surfaces to check for redirects: {len(all_misses)}')

redirect_hits = check_wiki_redirects(all_misses)

valid_redirect_hits = {}
for surface, target in redirect_hits.items():
    base_target = target.split('#')[0]
    if base_target in wiki_title_set or base_target.lower() in wiki_lower_lookup:
        valid_redirect_hits[surface] = target

print(f'Redirect hits:                 {len(redirect_hits)}')
print(f'Valid (land on known article): {len(valid_redirect_hits)}')

new_total = 543 + len(valid_redirect_hits)
print(f'Coverage after redirects: {new_total} / 1863 ({new_total/1863:.1%})')

print(f'\nAll redirect hits:')
for surface, target in valid_redirect_hits.items():
    typ = next((t for s, t in entity_list if s == surface), '???')
    print(f'  [{typ}] "{surface}" -> "{target}"')

Surfaces to check for redirects: 1318
Redirect hits:                 0
Valid (land on known article): 0
Coverage after redirects: 543 / 1863 (29.1%)

All redirect hits:


In [20]:
# CELL 18b — English Wikipedia redirect lookup for unresolved entities

WIKI_API_EN = 'https://en.wikipedia.org/w/api.php'

def check_wiki_redirects_en(surfaces: list[str], batch_size: int = 50) -> dict[str, str]:
    hits = {}

    for i in range(0, len(surfaces), batch_size):
        batch = surfaces[i:i + batch_size]
        params = {
            'action':    'query',
            'titles':    '|'.join(batch),
            'redirects': '1',
            'format':    'json',
        }
        try:
            resp = requests.get(
                WIKI_API_EN,
                params=params,
                headers=HEADERS,
                timeout=15
            )
            data = resp.json()
            query = data.get('query', {})

            # Redirects: input -> canonical
            for redirect in query.get('redirects', []):
                source = redirect['from']
                target = redirect['to']
                if source in batch:
                    hits[source] = target

            # Also catch direct hits — pages that exist without needing a redirect
            pages = query.get('pages', {})
            for page_id, page in pages.items():
                if page_id == '-1':
                    continue  # does not exist
                title = page.get('title')
                if title in batch:
                    hits[title] = title  # exists directly under this title

        except Exception as e:
            print(f'  Error on batch {i//batch_size}: {e}')

        time.sleep(0.2)

    return hits

# Build English Wikipedia title set for validation
# We already have the Irish Wikipedia title set (wiki_title_set)
# For English we just check whether the API returned a real page (page_id != -1)
# which the function above handles via the pages block

already_covered_all = (
    {s for s, t in hits_exact}
    | {s for s, t in hits_casefold}
    | set(demutation_hits.keys())
    | set(org_demutation_hits.keys())
    | {s for s, t, h, r in final_wd_keeps}
)

all_misses = [s for s, t in entity_list if s not in already_covered_all]
print(f'Surfaces to check against English Wikipedia: {len(all_misses)}')

en_redirect_hits = check_wiki_redirects_en(all_misses)

print(f'English Wikipedia hits (direct + redirect): {len(en_redirect_hits)}')

# Break down by entity type
for typ in ['PER', 'ORG', 'LOC']:
    type_surfaces = {s for s, t in entity_list if t == typ}
    type_hits = {s: t for s, t in en_redirect_hits.items() if s in type_surfaces}
    print(f'  {typ}: {len(type_hits)}')

new_total = 543 + len(en_redirect_hits)
print(f'\nCoverage after English Wikipedia: {new_total} / 1863 ({new_total/1863:.1%})')

print(f'\nAll English Wikipedia hits:')
for surface, target in sorted(en_redirect_hits.items()):
    typ = next((t for s, t in entity_list if s == surface), '???')
    match_type = 'redirect' if surface != target else 'direct'
    print(f'  [{typ}] "{surface}" -> "{target}" ({match_type})')

Surfaces to check against English Wikipedia: 1318
English Wikipedia hits (direct + redirect): 1064
  PER: 458
  ORG: 412
  LOC: 194

Coverage after English Wikipedia: 1607 / 1863 (86.3%)

All English Wikipedia hits:
  [ORG] "AIE" -> "AIE" (direct)
  [ORG] "ATU" -> "ATU" (direct)
  [PER] "Aaron O’Shea" -> "Aaron O’Shea" (direct)
  [LOC] "Aberdeen" -> "Aberdeen" (direct)
  [ORG] "Acadamh an Bhaile Meánach" -> "Acadamh an Bhaile Meánach" (direct)
  [LOC] "Aerfort Haneda" -> "Aerfort Haneda" (direct)
  [LOC] "Aerfort Idirnáisiúnta Thóiceo" -> "Aerfort Idirnáisiúnta Thóiceo" (direct)
  [LOC] "Aeriris" -> "Aeriris" (direct)
  [ORG] "Aga Khan" -> "Aga Khan" (direct)
  [PER] "Agent Orange" -> "Agent Orange" (direct)
  [PER] "Aiden Coffey" -> "Aiden Coffey" (direct)
  [PER] "Aindrias Mac Domhnaill" -> "Aindrias Mac Domhnaill" (direct)
  [PER] "Aire" -> "Aire" (direct)
  [PER] "Aire Airgeadais" -> "Aire Airgeadais" (direct)
  [PER] "Aire Dlí agus Cirt" -> "Aire Dlí agus Cirt" (direct)
  [PER] "A

In [21]:
# CELL 18c — audit English Wikipedia hit quality

WIKI_API_EN = 'https://en.wikipedia.org/w/api.php'

def get_page_info(titles: list[str], batch_size: int = 50) -> dict[str, dict]:
    """
    For each title, fetch page metadata: length, categories, whether it
    is a redirect, stub, or disambiguation page.
    Returns dict: title -> {'length', 'is_redirect', 'is_disambig', 'is_stub', 'categories'}
    """
    results = {}

    for i in range(0, len(titles), batch_size):
        batch = titles[i:i + batch_size]
        params = {
            'action':    'query',
            'titles':    '|'.join(batch),
            'prop':      'info|categories|pageprops',
            'inprop':    'length',
            'cllimit':   '5',
            'format':    'json',
        }
        try:
            resp = requests.get(
                WIKI_API_EN,
                params=params,
                headers=HEADERS,
                timeout=15
            )
            data = resp.json()
            pages = data.get('query', {}).get('pages', {})

            for page_id, page in pages.items():
                title  = page.get('title', '')
                length = page.get('length', 0)
                props  = page.get('pageprops', {})
                cats   = [c['title'] for c in page.get('categories', [])]

                is_disambig = 'disambiguation' in props or \
                              any('disambiguation' in c.lower() for c in cats)
                is_stub     = any('stub' in c.lower() for c in cats)
                is_redirect = page_id == '-1' or 'missing' in page

                results[title] = {
                    'length':      length,
                    'is_disambig': is_disambig,
                    'is_stub':     is_stub,
                    'is_redirect': is_redirect,
                    'categories':  cats,
                }

        except Exception as e:
            print(f'  Error on batch {i//batch_size}: {e}')

        time.sleep(0.2)

    return results

# Get the canonical targets from en_redirect_hits
canonical_targets = list(set(en_redirect_hits.values()))
print(f'Unique canonical targets to audit: {len(canonical_targets)}')

page_info = get_page_info(canonical_targets)

# Classify hits
STUB_THRESHOLD  = 3000   # bytes — below this, likely a stub
USEFUL_THRESHOLD = 8000  # bytes — above this, likely has real content

quality_counts = Counter()
useful_hits    = {}   # surface -> canonical target (real content)
stub_hits      = {}
disambig_hits  = {}

for surface, target in en_redirect_hits.items():
    base_target = target.split('#')[0]
    info = page_info.get(base_target) or page_info.get(target, {})

    if not info or info.get('is_redirect'):
        quality_counts['missing'] += 1
        continue

    if info.get('is_disambig'):
        quality_counts['disambig'] += 1
        disambig_hits[surface] = target
        continue

    length = info.get('length', 0)

    if length >= USEFUL_THRESHOLD:
        quality_counts['useful'] += 1
        useful_hits[surface] = target
    elif length >= STUB_THRESHOLD:
        quality_counts['stub_medium'] += 1
        stub_hits[surface] = target
    else:
        quality_counts['stub_short'] += 1

print(f'\n=== English Wikipedia hit quality ===')
for cat, count in quality_counts.most_common():
    print(f'  {cat:<20} {count:>5}  ({count/len(en_redirect_hits):.1%})')

print(f'\nUseful hits (>{USEFUL_THRESHOLD} bytes):  {len(useful_hits)}')
print(f'Medium stubs:                          {len(stub_hits)}')

# Coverage breakdown for useful hits only
useful_surfaces = set(useful_hits.keys())
useful_by_type  = Counter(t for s, t in entity_list if s in useful_surfaces)
print(f'\nUseful hit breakdown by type:')
for typ in ['PER', 'ORG', 'LOC']:
    type_total   = sum(1 for s, t in entity_list if t == typ)
    type_useful  = useful_by_type.get(typ, 0)
    print(f'  {typ}: {type_useful} / {type_total} ({type_useful/type_total:.1%})')

total_useful = 543 + len(useful_hits)
print(f'\nConservative coverage (useful only): {total_useful} / 1863 ({total_useful/1863:.1%})')
total_with_stubs = 543 + len(useful_hits) + len(stub_hits)
print(f'Coverage including medium stubs:     {total_with_stubs} / 1863 ({total_with_stubs/1863:.1%})')

print(f'\nSample useful hits:')
for s, t in list(useful_hits.items())[:15]:
    typ = next((et for es, et in entity_list if es == s), '???')
    print(f'  [{typ}] "{s}" -> "{t}"')

Unique canonical targets to audit: 1061

=== English Wikipedia hit quality ===
  missing                797  (74.9%)
  useful                 116  (10.9%)
  disambig                87  (8.2%)
  stub_medium             38  (3.6%)
  stub_short              26  (2.4%)

Useful hits (>8000 bytes):  116
Medium stubs:                          38

Useful hit breakdown by type:
  PER: 73 / 651 (11.2%)
  ORG: 34 / 654 (5.2%)
  LOC: 9 / 558 (1.6%)

Conservative coverage (useful only): 659 / 1863 (35.4%)
Coverage including medium stubs:     697 / 1863 (37.4%)

Sample useful hits:
  [ORG] "An Roinn Oideachais agus Eolaíochta" -> "Department of Education and Youth"
  [LOC] "Aberdeen" -> "Aberdeen"
  [ORG] "Aga Khan" -> "Aga Khan"
  [PER] "Agent Orange" -> "Agent Orange"
  [PER] "Akhil Sharma" -> "Akhil Sharma"
  [PER] "Ali" -> "Ali"
  [PER] "Anne-Marie" -> "Anne-Marie"
  [PER] "Beverley Cooper-Flynn" -> "Beverley Flynn"
  [ORG] "BCE" -> "Common Era"
  [ORG] "Barnardos" -> "Barnardo's"
  [PER] "Aodh 

In [22]:
# CELL 19 — English back-translation + DBpedia lookup for ORG misses

DBPEDIA_SPARQL = 'https://dbpedia.org/sparql'

# Rule-based Irish -> English prefix translation for government/institutional ORGs
TRANSLATION_RULES = [
    (r'^An Roinn\s+',           'Department of '),
    (r'^An Chomhairle\s+',      'Council of '),
    (r'^An Coiste\s+',          'Committee on '),
    (r'^An Coimisiún\s+',       'Commission on '),
    (r'^An Bord\s+',            'Board of '),
    (r'^An tÚdarás\s+',         'Authority for '),
    (r'^Údarás\s+',             'Authority for '),
    (r'^An tOifig\s+',          'Office of '),
    (r'^Oifig\s+',              'Office of '),
    (r'^An Institiúid\s+',      'Institute of '),
    (r'^Institiúid\s+',         'Institute of '),
    (r'^Ollscoil\s+',           'University of '),
    (r'^Coláiste\s+',           'College of '),
    (r'^Scoil\s+',              'School of '),
    (r'^Cumann\s+',             'Association of '),
    (r'^Ciste\s+',              'Fund for '),
    (r'^Lárionad\s+',           'Centre for '),
    (r'^Ionad\s+',              'Centre for '),
    (r'^Foras\s+',              'Institute for '),
    (r'^Comhlacht\s+',          'Body for '),
    (r'^Banc\s+',               'Bank of '),
    (r'^Páirtí\s+',             'Party of '),
    (r'^Arm\s+',                'Army of '),
    (r'^Cúirt\s+',              'Court of '),
]

# Common Irish word translations for the remainder of the string
WORD_TRANSLATIONS = {
    'na': 'of the', 'an': 'the', 'agus': 'and', 'do': 'for',
    'le': 'for', 'i': 'in', 'um': 'for', 'faoi': 'under',
    'Oideachas': 'Education', 'Oideachais': 'Education',
    'Sláinte': 'Health', 'Iompair': 'Transport',
    'Airgeadais': 'Finance', 'Dlí': 'Law', 'Cirt': 'Justice',
    'Gnóthaí': 'Affairs', 'Ealaíon': 'Arts', 'Eolaíochta': 'Science',
    'Nuálaíochta': 'Innovation', 'Teicneolaíochta': 'Technology',
    'Fiontar': 'Enterprise', 'Fostaíochta': 'Employment',
    'Tithíochta': 'Housing', 'Comhshaoil': 'Environment',
    'Talmhaíochta': 'Agriculture', 'Cosanta': 'Defence',
    'Turasóireachta': 'Tourism', 'Spóirt': 'Sport',
    'Cultúir': 'Culture', 'Gaeltachta': 'Gaeltacht',
    'Pobail': 'Community', 'Tuaithe': 'Rural',
    'Éireann': 'Ireland', 'Náisiúnta': 'National',
    'Idirnáisiúnta': 'International', 'Cathrach': 'City',
    'Contae': 'County', 'Réigiúnach': 'Regional',
}

def translate_irish_org(surface: str) -> str | None:
    """
    Attempt rule-based translation of Irish institutional name to English.
    Returns English candidate string or None if no rule matches.
    """
    translated = surface
    matched_prefix = False

    for pattern, replacement in TRANSLATION_RULES:
        if re.match(pattern, translated, re.IGNORECASE):
            translated = re.sub(pattern, replacement, translated, count=1, flags=re.IGNORECASE)
            matched_prefix = True
            break

    if not matched_prefix:
        return None

    # Translate remaining words
    words = translated.split()
    words = [WORD_TRANSLATIONS.get(w, w) for w in words]
    translated = ' '.join(words)

    return translated.strip()

def query_dbpedia_label(english_name: str) -> dict | None:
    """
    Search DBpedia for an entity by English label.
    Returns first match with type and abstract or None.
    """
    query = f"""
SELECT ?item ?label ?type ?abstract WHERE {{
  ?item rdfs:label "{english_name}"@en .
  OPTIONAL {{ ?item rdf:type ?type . }}
  OPTIONAL {{ ?item dbo:abstract ?abstract .
              FILTER (lang(?abstract) = 'en') }}
  SERVICE wikibase:label {{
    bd:serviceParam wikibase:language "en" .
  }}
}}
LIMIT 5
"""
    # Use DBpedia's own SPARQL endpoint syntax
    query_simple = f"""
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?item ?type ?abstract WHERE {{
  ?item rdfs:label "{english_name}"@en .
  OPTIONAL {{ ?item rdf:type ?type . }}
  OPTIONAL {{ ?item dbo:abstract ?abstract .
              FILTER (lang(?abstract) = 'en') }}
}}
LIMIT 3
"""
    try:
        resp = requests.get(
            DBPEDIA_SPARQL,
            params={'query': query_simple, 'format': 'application/sparql-results+json'},
            headers=HEADERS,
            timeout=15
        )
        if resp.status_code != 200:
            return None
        bindings = resp.json()['results']['bindings']
        if not bindings:
            return None
        first = bindings[0]
        return {
            'uri':      first['item']['value'],
            'type':     first.get('type', {}).get('value', ''),
            'abstract': first.get('abstract', {}).get('value', '')[:150],
        }
    except Exception:
        return None

# Run on ORG misses only — that is where translation rules apply
org_miss_surfaces = [s for s in org_demutation_miss]
print(f'ORG misses to attempt translation: {len(org_miss_surfaces)}')

translation_hits   = {}
translation_misses = []
dbpedia_hits       = {}

for surface in org_miss_surfaces:
    english = translate_irish_org(surface)
    if english:
        translation_hits[surface] = english
    else:
        translation_misses.append(surface)

print(f'Surfaces with translation rule match: {len(translation_hits)}')
print(f'No rule matched:                      {len(translation_misses)}')

print(f'\nSample translations:')
for s, e in list(translation_hits.items())[:20]:
    print(f'  "{s}"')
    print(f'    -> "{e}"')

# Now query DBpedia for each translated string
print(f'\nQuerying DBpedia for {len(translation_hits)} translated ORG names...')

for i, (surface, english) in enumerate(translation_hits.items()):
    result = query_dbpedia_label(english)
    if result:
        dbpedia_hits[surface] = {
            'english_name': english,
            'uri':          result['uri'],
            'type':         result['type'],
            'abstract':     result['abstract'],
        }
    time.sleep(0.3)
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(translation_hits)} queried, {len(dbpedia_hits)} hits')

print(f'\nDBpedia hits: {len(dbpedia_hits)} / {len(translation_hits)}')

new_total = 543 + len(dbpedia_hits)
print(f'Coverage after DBpedia back-translation: {new_total} / 1863 ({new_total/1863:.1%})')

print(f'\nSample DBpedia hits:')
for s, r in list(dbpedia_hits.items())[:15]:
    print(f'  "{s}"')
    print(f'    -> "{r["english_name"]}"')
    print(f'    -> {r["uri"]}')
    print(f'    -> {r["abstract"][:100]}')

ORG misses to attempt translation: 448
Surfaces with translation rule match: 41
No rule matched:                      407

Sample translations:
  "An Chomhairle Ealaíon"
    -> "Council of Arts"
  "An Roinn Ceapachán agus Forbairt Foirne"
    -> "Department of Ceapachán and Forbairt Foirne"
  "An Roinn Gnóthaí Pobail Tuaithe agus Gaeltachta"
    -> "Department of Affairs Community Rural and Gaeltacht"
  "An Roinn Nuálaíochta agus Teicneolaíochta"
    -> "Department of Innovation and Technology"
  "An Roinn Oideachais agus Eolaíochta"
    -> "Department of Education and Science"
  "Arm Cathartha"
    -> "Army of Cathartha"
  "Arm Slánaithe"
    -> "Army of Slánaithe"
  "Ciste Sóisialta na hEorpa"
    -> "Fund for Sóisialta of the hEorpa"
  "Coláiste Ghobnatan"
    -> "College of Ghobnatan"
  "Cumann Báire Setanta"
    -> "Association of Báire Setanta"
  "Cumann Cúramóirí na hÉireann"
    -> "Association of Cúramóirí of the hÉireann"
  "Cumann Fíoncheannaithe Bhaile Átha Cliath"
    -> "

In [23]:
# CELL 20 — English Wikipedia quality filter via Wikidata sitelinks
#
# For each surface in useful_hits (117) and stub_hits (38), retrieve the
# Wikidata QID via the EN-Wikipedia sitelinks API, then fetch description
# and instance_of. Run score_hit_v2 to accept or reject. This combines
# the quality filter and sitelinks enrichment into a single pass.
#
# Input:  useful_hits, stub_hits  (surface -> canonical EN-Wiki title)
#         entity_list             (surface, type tuples)
#         score_hit_v2            (scoring function from Cell 16)
# Output: en_wiki_clean           dict: surface -> {qid, title, description, instance_of}
#         en_wiki_clean_surfaces  set of accepted surface forms
 
import time, requests
 
WIKI_API_EN  = 'https://en.wikipedia.org/w/api.php'
WD_API       = 'https://www.wikidata.org/w/api.php'
HEADERS      = {'User-Agent': 'IrishNER-PhaseC/1.0 (MSc dissertation, TU Dublin)'}
 
entity_type_map = {s: t for s, t in entity_list}
 
def get_wikidata_qid_batch(titles: list[str], batch_size: int = 50) -> dict[str, str]:
    """
    Map EN-Wikipedia titles -> Wikidata QIDs via the pageprops API.
    Returns dict: title -> QID (or empty string if not found).
    """
    qid_map = {}
    for i in range(0, len(titles), batch_size):
        batch = titles[i:i + batch_size]
        params = {
            'action':  'query',
            'titles':  '|'.join(batch),
            'prop':    'pageprops',
            'ppprop':  'wikibase_item',
            'format':  'json',
        }
        try:
            resp = requests.get(WIKI_API_EN, params=params, headers=HEADERS, timeout=15)
            pages = resp.json().get('query', {}).get('pages', {})
            for page in pages.values():
                title = page.get('title', '')
                qid   = page.get('pageprops', {}).get('wikibase_item', '')
                if title and qid:
                    qid_map[title] = qid
        except Exception as e:
            print(f'  QID batch error (batch {i // batch_size}): {e}')
        time.sleep(0.15)
    return qid_map
 
 
def get_wikidata_descriptions_batch(qids: list[str], batch_size: int = 50) -> dict[str, dict]:
    """
    For a list of QIDs, fetch description and P31 (instance_of) label.
    Returns dict: QID -> {description, instance_of}
    """
    desc_map = {}
    for i in range(0, len(qids), batch_size):
        batch = qids[i:i + batch_size]
        params = {
            'action':   'wbgetentities',
            'ids':      '|'.join(batch),
            'props':    'descriptions|claims',
            'languages':'en',
            'format':   'json',
        }
        try:
            resp = requests.get(WD_API, params=params, headers=HEADERS, timeout=20)
            entities = resp.json().get('entities', {})
            for qid, ent in entities.items():
                desc = ent.get('descriptions', {}).get('en', {}).get('value', '')
                # P31 = instance of; take first value's label if present
                p31_claims = ent.get('claims', {}).get('P31', [])
                instance_of = ''
                if p31_claims:
                    p31_val = p31_claims[0].get('mainsnak', {}) \
                                           .get('datavalue', {}) \
                                           .get('value', {})
                    p31_qid = p31_val.get('id', '') if isinstance(p31_val, dict) else ''
                    # Resolve the P31 QID label in the same batch where possible;
                    # fall back to empty string — the description alone is usually sufficient.
                    instance_of = p31_qid
                desc_map[qid] = {'description': desc, 'instance_of': instance_of}
        except Exception as e:
            print(f'  Description batch error (batch {i // batch_size}): {e}')
        time.sleep(0.15)
    return desc_map
 
 
# --- Step 1: collect all candidate surfaces (useful + medium stubs) ---
candidates = {}   # surface -> canonical_title
for surface, title in useful_hits.items():
    candidates[surface] = title.split('#')[0]
for surface, title in stub_hits.items():
    if surface not in candidates:
        candidates[surface] = title.split('#')[0]
 
print(f'Candidates for EN-Wiki quality filter: {len(candidates)}')
 
# --- Step 2: fetch QIDs for all canonical titles ---
unique_titles = list(set(candidates.values()))
print(f'Unique canonical titles to QID-lookup: {len(unique_titles)}')
 
title_to_qid = get_wikidata_qid_batch(unique_titles)
print(f'QIDs retrieved: {len(title_to_qid)} / {len(unique_titles)}')
 
# --- Step 3: fetch descriptions for all QIDs ---
qids_to_fetch = list(set(title_to_qid.values()))
print(f'Fetching descriptions for {len(qids_to_fetch)} QIDs...')
 
qid_desc_map = get_wikidata_descriptions_batch(qids_to_fetch)
print(f'Descriptions retrieved: {len(qid_desc_map)}')
 
# --- Step 4: score each candidate with score_hit_v2 ---
en_wiki_clean   = {}   # surface -> {qid, title, description, instance_of, verdict_reason}
en_wiki_discard = {}
 
for surface, title in candidates.items():
    entity_type = entity_type_map.get(surface, 'UNK')
    qid         = title_to_qid.get(title, '')
    if not qid:
        en_wiki_discard[surface] = {'reason': 'no QID found'}
        continue
 
    desc_info = qid_desc_map.get(qid, {})
    hit = {
        'qid':         qid,
        'description': desc_info.get('description', ''),
        'instance_of': desc_info.get('instance_of', ''),
    }
 
    verdict, reason = score_hit_v2(surface, entity_type, hit)
 
    if verdict == 'keep':
        en_wiki_clean[surface] = {
            'qid':           qid,
            'title':         title,
            'description':   hit['description'],
            'instance_of':   hit['instance_of'],
            'verdict_reason': reason,
        }
    else:
        en_wiki_discard[surface] = {'reason': reason, 'qid': qid, 'desc': hit['description'][:60]}
 
en_wiki_clean_surfaces = set(en_wiki_clean.keys())
 
print(f'\n=== EN-Wikipedia quality filter results ===')
print(f'  Accepted: {len(en_wiki_clean)}')
print(f'  Rejected: {len(en_wiki_discard)}')
 
# Breakdown by entity type
for typ in ['PER', 'ORG', 'LOC']:
    count = sum(1 for s in en_wiki_clean if entity_type_map.get(s) == typ)
    print(f'  {typ}: {count}')
 
# Running coverage total
prior_clean    = 543   # from Cell 17 + Cell 19 (Toronto hit dropped as false positive)
new_clean_total = prior_clean + len(en_wiki_clean)
print(f'\nCoverage after EN-Wiki filter: {new_clean_total} / 1863 ({new_clean_total / 1863:.1%})')
 
print(f'\nSample accepted:')
for s, r in list(en_wiki_clean.items())[:15]:
    typ = entity_type_map.get(s, '???')
    print(f'  [{typ}] "{s}" -> {r["qid"]} | {r["description"][:65]}')
 
print(f'\nSample rejected:')
for s, r in list(en_wiki_discard.items())[:10]:
    typ = entity_type_map.get(s, '???')
    print(f'  [{typ}] "{s}" | {r["reason"]} | {r.get("desc", "")}')
 

Candidates for EN-Wiki quality filter: 154
Unique canonical titles to QID-lookup: 151
QIDs retrieved: 151 / 151
Fetching descriptions for 151 QIDs...
Descriptions retrieved: 151

=== EN-Wikipedia quality filter results ===
  Accepted: 72
  Rejected: 82
  PER: 42
  ORG: 23
  LOC: 7

Coverage after EN-Wiki filter: 615 / 1863 (33.0%)

Sample accepted:
  [ORG] "An Roinn Oideachais agus Eolaíochta" -> Q4294533 | Irish government department
  [LOC] "Aberdeen" -> Q36405 | city in Scotland, United Kingdom
  [PER] "Akhil Sharma" -> Q4700740 | Indian American writer
  [PER] "Anne-Marie" -> Q25396976 | British singer (born 1991)
  [PER] "Beverley Cooper-Flynn" -> Q4899405 | Irish politician
  [ORG] "Barnardos" -> Q2884670 | British charity
  [ORG] "CETA" -> Q1121969 | free-trade agreement between Canada and the European Union
  [ORG] "Buachaillí Bána" -> Q862527 | secret Irish agrarian organisation in 18th-century Ireland
  [LOC] "Bornholm" -> Q769680 | Danish island
  [PER] "Brian McEniff" -> Q4

In [24]:

# CELL 21 — Oireachtas API PER coverage
#
# Fetches all current and historical Oireachtas members (TDs and senators)
# from the public REST API. Matches against PER misses using exact,
# casefold, and demutation comparison.
#
# Input:  entity_list, already_covered_all, en_wiki_clean_surfaces
# Output: oireachtas_hits  dict: surface -> {uri, full_name, house}
 
import unicodedata, re
 
OIREACHTAS_API = 'https://api.oireachtas.ie/v1'
HEADERS_OIR    = {
    'Accept':     'application/json',
    'User-Agent': 'IrishNER-PhaseC/1.0 (MSc dissertation, TU Dublin)',
}
 
# Mutation helpers (reuse pattern from earlier cells)
LENITION_MAP = {
    'bh': 'b', 'ch': 'c', 'dh': 'd', 'fh': 'f', 'gh': 'g',
    'mh': 'm', 'ph': 'p', 'sh': 's', 'th': 't',
}
ECLIPSIS_MAP = {
    'mb': 'b', 'gc': 'c', 'nd': 'd', 'bhf': 'f', 'ng': 'g',
    'bp': 'p', 'dt': 't',
}
 
def demutate(token: str) -> str:
    t = token.lower()
    for mut, base in ECLIPSIS_MAP.items():
        if t.startswith(mut):
            return base + token[len(mut):]
    for mut, base in LENITION_MAP.items():
        if t.startswith(mut):
            return base + token[len(mut):]
    return token
 
def normalize_name(name: str) -> str:
    """Casefold + strip accents for fuzzy comparison."""
    nfkd = unicodedata.normalize('NFKD', name.casefold())
    return ''.join(c for c in nfkd if not unicodedata.combining(c))
 
def fetch_oireachtas_members(date_start: str = '1919-01-01') -> list[dict]:
    """
    Fetch all members from the Oireachtas API paged endpoint.
    Returns list of {uri, full_name, normalized_name}.
    """
    members  = []
    limit    = 50
    skip     = 0
    endpoint = f'{OIREACHTAS_API}/members'
 
    while True:
        params = {
            'date_start': date_start,
            'limit':      limit,
            'skip':       skip,
            'format':     'json',
        }
        try:
            resp = requests.get(endpoint, params=params, headers=HEADERS_OIR, timeout=20)
            if resp.status_code != 200:
                print(f'  API error {resp.status_code} at skip={skip}')
                break
            data    = resp.json()
            results = data.get('results', [])
            if not results:
                break
            for item in results:
                member = item.get('member', {})
                name   = member.get('fullName', '') or member.get('lastName', '')
                uri    = member.get('uri', '')
                if name:
                    members.append({
                        'uri':             uri,
                        'full_name':       name,
                        'normalized_name': normalize_name(name),
                    })
            skip += limit
            if len(results) < limit:
                break
            time.sleep(0.1)
        except Exception as e:
            print(f'  Fetch error at skip={skip}: {e}')
            break
 
    return members
 
 
print('Fetching Oireachtas members...')
oireachtas_members = fetch_oireachtas_members()
print(f'Total members fetched: {len(oireachtas_members)}')
 
# Build lookup: normalized_name -> member record
oir_lookup = {}
for m in oireachtas_members:
    oir_lookup[m['normalized_name']] = m
 
# Identify PER misses not yet covered
covered_so_far = (
    already_covered_all
    | en_wiki_clean_surfaces
)
per_misses = [s for s, t in entity_list if t == 'PER' and s not in covered_so_far]
print(f'PER misses to check against Oireachtas: {len(per_misses)}')
 
# Match: exact casefold, then accent-stripped, then first-token demutation
oireachtas_hits   = {}
oireachtas_misses = []
 
for surface in per_misses:
    norm_surface = normalize_name(surface)
 
    # Pass 1: direct normalized match
    if norm_surface in oir_lookup:
        oireachtas_hits[surface] = oir_lookup[norm_surface]
        continue
 
    # Pass 2: demutate first token of surface
    tokens    = surface.split()
    demutated = ' '.join([demutate(tokens[0])] + tokens[1:])
    norm_dem  = normalize_name(demutated)
    if norm_dem in oir_lookup:
        oireachtas_hits[surface] = oir_lookup[norm_dem]
        continue
 
    # Pass 3: partial — surface is a substring of a member name
    # (handles "Ó Murchú" matching "Seán Ó Murchú")
    matched_partial = None
    for norm_name, member in oir_lookup.items():
        if norm_surface in norm_name and len(norm_surface) > 5:
            matched_partial = member
            break
    if matched_partial:
        oireachtas_hits[surface] = matched_partial
        continue
 
    oireachtas_misses.append(surface)
 
oireachtas_surfaces = set(oireachtas_hits.keys())
 
print(f'\n=== Oireachtas PER matches ===')
print(f'  Matched: {len(oireachtas_hits)}')
print(f'  Unmatched: {len(oireachtas_misses)}')
 
running_total = prior_clean + len(en_wiki_clean) + len(oireachtas_hits)
print(f'\nCoverage after Oireachtas: {running_total} / 1863 ({running_total / 1863:.1%})')
 
print(f'\nSample matches:')
for s, m in list(oireachtas_hits.items())[:20]:
    print(f'  "{s}" -> "{m["full_name"]}" | {m["uri"]}')

Fetching Oireachtas members...
Total members fetched: 1928
PER misses to check against Oireachtas: 447

=== Oireachtas PER matches ===
  Matched: 41
  Unmatched: 406

Coverage after Oireachtas: 656 / 1863 (35.2%)

Sample matches:
  "Andrews" -> "Barry Andrews" | https://data.oireachtas.ie/ie/oireachtas/member/id/Barry-Andrews.D.2002-06-06
  "Anthony" -> "Richard Sidney Anthony" | https://data.oireachtas.ie/ie/oireachtas/member/id/Richard-Sidney-Anthony.D.1927-06-23
  "Brendan" -> "Brendan Corish" | https://data.oireachtas.ie/ie/oireachtas/member/id/Brendan-Corish.D.1945-12-04
  "Butler" -> "Bernard Butler" | https://data.oireachtas.ie/ie/oireachtas/member/id/Bernard-Butler.D.1943-07-01
  "Ciarán" -> "Ciarán Ahern" | https://data.oireachtas.ie/ie/oireachtas/member/id/Ciarán-Ahern.D.2024-11-29
  "Collins" -> "Con Collins" | https://data.oireachtas.ie/ie/oireachtas/member/id/Conor-Collins.D.1919-01-21
  "Comiskey" -> "Michael Comiskey" | https://data.oireachtas.ie/ie/oireachtas/member/id/

In [25]:

# CELL 22 — Wikidata fuzzy search for remaining PER and ORG misses
#
# Runs wbsearchentities on PER and ORG surfaces not yet covered by any
# prior source. Applies score_hit_v2 to filter noise. Skips LOC — the
# GeoNames and Irish Wikipedia passes are the appropriate LOC sources.
#
# Input:  entity_list, covered_so_far (updated), oireachtas_surfaces
#         score_hit_v2, WD_API, HEADERS
# Output: wd_fuzzy_hits  dict: surface -> {qid, description, instance_of}
 
def wikidata_fuzzy_search(surface: str, entity_type: str,
                          language: str = 'en', limit: int = 3) -> list[dict]:
    """
    Search Wikidata for a surface form using wbsearchentities.
    Returns up to `limit` candidates with qid, description, instance_of stub.
    """
    params = {
        'action':   'wbsearchentities',
        'search':   surface,
        'language': language,
        'limit':    limit,
        'format':   'json',
    }
    try:
        resp = requests.get(WD_API, params=params, headers=HEADERS, timeout=15)
        results = resp.json().get('search', [])
        candidates = []
        for r in results:
            candidates.append({
                'qid':         r.get('id', ''),
                'label':       r.get('label', ''),
                'description': r.get('description', ''),
                'instance_of': '',   # not returned by wbsearchentities; scoring uses description only
            })
        return candidates
    except Exception:
        return []
 
 
# Build the full covered set including all sources up to Cell 21
covered_after_21 = covered_so_far | oireachtas_surfaces
 
per_org_misses = [
    (s, t) for s, t in entity_list
    if t in ('PER', 'ORG') and s not in covered_after_21
]
print(f'PER + ORG misses for Wikidata fuzzy search: {len(per_org_misses)}')
 
wd_fuzzy_hits   = {}
wd_fuzzy_nofit  = []
 
for i, (surface, entity_type) in enumerate(per_org_misses):
    candidates = wikidata_fuzzy_search(surface, entity_type)
 
    accepted = False
    for cand in candidates:
        verdict, reason = score_hit_v2(surface, entity_type, cand)
        if verdict == 'keep':
            wd_fuzzy_hits[surface] = {
                'qid':           cand['qid'],
                'label':         cand['label'],
                'description':   cand['description'],
                'verdict_reason': reason,
            }
            accepted = True
            break
 
    if not accepted:
        wd_fuzzy_nofit.append(surface)
 
    time.sleep(0.12)
    if (i + 1) % 100 == 0:
        print(f'  {i + 1}/{len(per_org_misses)} searched, {len(wd_fuzzy_hits)} hits so far')
 
wd_fuzzy_surfaces = set(wd_fuzzy_hits.keys())
 
print(f'\n=== Wikidata fuzzy search results ===')
print(f'  Accepted: {len(wd_fuzzy_hits)}')
print(f'  No fit:   {len(wd_fuzzy_nofit)}')
 
for typ in ['PER', 'ORG']:
    count = sum(1 for s in wd_fuzzy_hits if entity_type_map.get(s) == typ)
    print(f'  {typ}: {count}')
 
running_total = prior_clean + len(en_wiki_clean) + len(oireachtas_hits) + len(wd_fuzzy_hits)
print(f'\nCoverage after fuzzy search: {running_total} / 1863 ({running_total / 1863:.1%})')
 
print(f'\nSample accepted (PER):')
for s, r in [(s, r) for s, r in wd_fuzzy_hits.items() if entity_type_map.get(s) == 'PER'][:10]:
    print(f'  "{s}" -> {r["qid"]} | {r["description"][:65]}')
 
print(f'\nSample accepted (ORG):')
for s, r in [(s, r) for s, r in wd_fuzzy_hits.items() if entity_type_map.get(s) == 'ORG'][:10]:
    print(f'  "{s}" -> {r["qid"]} | {r["description"][:65]}')
 

PER + ORG misses for Wikidata fuzzy search: 887
  100/887 searched, 2 hits so far
  200/887 searched, 2 hits so far
  300/887 searched, 4 hits so far
  400/887 searched, 10 hits so far
  500/887 searched, 11 hits so far
  600/887 searched, 13 hits so far
  700/887 searched, 14 hits so far
  800/887 searched, 14 hits so far

=== Wikidata fuzzy search results ===
  Accepted: 14
  No fit:   873
  PER: 7
  ORG: 7

Coverage after fuzzy search: 670 / 1863 (36.0%)

Sample accepted (PER):
  "Edward Hay" -> Q75323257 | (1722-1779)
  "John Downey" -> Q112029959 | irish-born American politician (1834–1906)
  "John Ó Donnell" -> Q106889242 | politician in Massachusetts, US (b. 1853)
  "Johnny Kelly" -> Q126697547 | Irish hurling manager, coach and former player
  "Joyce" -> Q101423005 | (1608-1687)
  "Mustafa Barzaní" -> Q368858 | Kurdish nationalist (1903–1979)
  "Siobhán Kelly" -> Q76227535 | (born 1952)

Sample accepted (ORG):
  "AIE" -> Q4204162 | Society of Performing Artists of Spain
  "ATU"

In [26]:

# CELL 23 — Phase C coverage assessment and go/no-go decision
#
# Consolidates all coverage sources into a single definitive count.
# Computes final coverage rate and documents the empirical ceiling finding.
 
# --- Assemble the definitive covered set ---
phase_c_covered = (
    already_covered_all       # Cell 17: Irish Wikipedia + demutation + clean Wikidata (543)
    | en_wiki_clean_surfaces  # Cell 20: EN-Wikipedia via sitelinks quality filter
    | oireachtas_surfaces     # Cell 21: Oireachtas API PER
    | wd_fuzzy_surfaces       # Cell 22: Wikidata fuzzy search
)
 
total_covered = len(phase_c_covered)
total_entities = len(entity_list)
coverage_rate  = total_covered / total_entities
 
# --- Breakdown by source (additive, not overlapping) ---
irish_wiki_base   = 543   # Cells 10–17: Irish Wikipedia + GeoNames + Wikidata
en_wiki_new       = len(en_wiki_clean_surfaces - already_covered_all)
oireachtas_new    = len(oireachtas_surfaces - already_covered_all - en_wiki_clean_surfaces)
fuzzy_new         = len(wd_fuzzy_surfaces - already_covered_all - en_wiki_clean_surfaces - oireachtas_surfaces)
 
print('=== Phase C coverage: definitive count ===')
print(f'')
print(f'Irish Wikipedia + demutation + Wikidata:  {irish_wiki_base:>4}  ({irish_wiki_base / total_entities:.1%})')
print(f'English Wikipedia (sitelinks filter):    +{en_wiki_new:>4}  ({en_wiki_new / total_entities:.1%})')
print(f'Oireachtas API (PER):                    +{oireachtas_new:>4}  ({oireachtas_new / total_entities:.1%})')
print(f'Wikidata fuzzy search (PER + ORG):       +{fuzzy_new:>4}  ({fuzzy_new / total_entities:.1%})')
print(f'                                          ----')
print(f'Total covered:                            {total_covered:>4}  ({coverage_rate:.1%})')
print(f'Irreducible misses:                       {total_entities - total_covered:>4}  ({(total_entities - total_covered) / total_entities:.1%})')
 
# --- Breakdown by entity type ---
print(f'\n--- Coverage by entity type ---')
for typ in ['PER', 'ORG', 'LOC']:
    type_total   = sum(1 for s, t in entity_list if t == typ)
    type_covered = sum(1 for s, t in entity_list if t == typ and s in phase_c_covered)
    print(f'  {typ}: {type_covered} / {type_total} ({type_covered / type_total:.1%})')
 
# --- Go/no-go decision ---
print(f'\n--- Go/no-go ---')
if coverage_rate >= 0.40:
    print(f'Coverage {coverage_rate:.1%} >= 40% threshold.')
    print('Proceed to triple construction (Cell 24).')
    print('Covered-subset analysis will have adequate statistical power.')
else:
    print(f'Coverage {coverage_rate:.1%} < 40% threshold.')
    print('Documenting as empirical ceiling: public KB coverage is structurally')
    print('capped for literary/journalistic Irish text regardless of KB combination.')
    print('Proceeding to triple construction — the ceiling finding directly addresses')
    print('the subsidiary RQ and constitutes a novel empirical contribution.')
    print('Covered-subset analysis proceeds on the available covered population.')

=== Phase C coverage: definitive count ===

Irish Wikipedia + demutation + Wikidata:   543  (29.1%)
English Wikipedia (sitelinks filter):    +  72  (3.9%)
Oireachtas API (PER):                    +  41  (2.2%)
Wikidata fuzzy search (PER + ORG):       +  14  (0.8%)
                                          ----
Total covered:                             672  (36.1%)
Irreducible misses:                       1191  (63.9%)

--- Coverage by entity type ---
  PER: 252 / 651 (38.7%)
  ORG: 180 / 654 (27.5%)
  LOC: 240 / 558 (43.0%)

--- Go/no-go ---
Coverage 36.1% < 40% threshold.
Documenting as empirical ceiling: public KB coverage is structurally
capped for literary/journalistic Irish text regardless of KB combination.
Proceeding to triple construction — the ceiling finding directly addresses
the subsidiary RQ and constitutes a novel empirical contribution.
Covered-subset analysis proceeds on the available covered population.


In [27]:
# CELL 23b — post-coverage audit: remove false positives before triple construction
#
# The Oireachtas partial-match pass (Pass 3) and single-token matches
# are too permissive. Single-token surfaces that are common given names
# or surnames cannot be reliably attributed to a specific member.
# Several Wikidata fuzzy hits are clearly wrong-corpus entities.
# This cell removes confirmed false positives and recomputes the
# definitive pre-construction coverage count.
#
# Input:  oireachtas_hits, wd_fuzzy_hits, phase_c_covered
# Output: oireachtas_clean, wd_fuzzy_clean, phase_c_covered_clean (final)

# --- Oireachtas: reject single-token surfaces and known wrong matches ---
# Single-token surfaces are common names that cannot be attributed to a
# specific member without a full-name match. Reject all surfaces where
# the surface is a single whitespace-free token AND the match was via
# partial (Pass 3) or casefold rather than exact full-name match.

# Explicit false positives identified from output inspection
OIREACHTAS_FALSE_POSITIVES = {
    'Anthony',    # common given name; matched Richard Sidney Anthony (1927)
    'Berry',      # matched via partial; wrong referent
    'Brendan',    # common given name; matched Brendan Corish
    'Ciarán',     # common given name; matched Ciarán Ahern (2024)
    'Collins',    # surname; matched Con Collins — too common to attribute
    'Donncha',    # common given name; matched Donnchadh Ó Briain
    'Dónall',     # matched via casefold; given name only
    'Fidelma',    # common given name
    'Garret',     # common given name; matched Garret Ahearn
    'John T',     # partial match artefact; "John T" is not a person surface
}

# Additionally, reject any remaining single-token surface where the token
# is 7 characters or fewer — these are too short to be unambiguous.
single_token_short = {
    s for s in oireachtas_hits
    if len(s.split()) == 1 and len(s) <= 7
    and s not in {'De Róiste'}   # compound surface, keep
}

oireachtas_reject = OIREACHTAS_FALSE_POSITIVES | single_token_short

oireachtas_clean = {
    s: m for s, m in oireachtas_hits.items()
    if s not in oireachtas_reject
}

print(f'Oireachtas hits before audit: {len(oireachtas_hits)}')
print(f'Rejected (false positives):   {len(oireachtas_hits) - len(oireachtas_clean)}')
print(f'Clean Oireachtas hits:        {len(oireachtas_clean)}')

print(f'\nRejected surfaces:')
for s in sorted(oireachtas_reject & set(oireachtas_hits.keys())):
    m = oireachtas_hits[s]
    print(f'  "{s}" was -> "{m["full_name"]}"')

print(f'\nRetained Oireachtas matches:')
for s, m in sorted(oireachtas_clean.items()):
    print(f'  "{s}" -> "{m["full_name"]}" | {m["uri"]}')

# --- Wikidata fuzzy: remove wrong-corpus entities ---
WD_FUZZY_FALSE_POSITIVES = {
    'AIE',   # Society of Performing Artists of Spain — wrong corpus
    'ATU',   # University in Arkansas — wrong corpus
}

wd_fuzzy_clean = {
    s: r for s, r in wd_fuzzy_hits.items()
    if s not in WD_FUZZY_FALSE_POSITIVES
}

print(f'\nWikidata fuzzy hits before audit: {len(wd_fuzzy_hits)}')
print(f'Rejected:                         {len(wd_fuzzy_hits) - len(wd_fuzzy_clean)}')
print(f'Clean fuzzy hits:                 {len(wd_fuzzy_clean)}')

print(f'\nRetained fuzzy matches:')
for s, r in wd_fuzzy_clean.items():
    typ = entity_type_map.get(s, '???')
    print(f'  [{typ}] "{s}" -> {r["qid"]} | {r["description"][:65]}')

# --- Recompute definitive covered set ---
oireachtas_clean_surfaces = set(oireachtas_clean.keys())
wd_fuzzy_clean_surfaces   = set(wd_fuzzy_clean.keys())

phase_c_covered_clean = (
    already_covered_all
    | en_wiki_clean_surfaces
    | oireachtas_clean_surfaces
    | wd_fuzzy_clean_surfaces
)

total_covered_clean = len(phase_c_covered_clean)
coverage_rate_clean = total_covered_clean / total_entities

# Additive breakdown (non-overlapping)
irish_wiki_base  = 543
en_wiki_new      = len(en_wiki_clean_surfaces - already_covered_all)
oireachtas_new   = len(oireachtas_clean_surfaces - already_covered_all - en_wiki_clean_surfaces)
fuzzy_new        = len(wd_fuzzy_clean_surfaces
                       - already_covered_all
                       - en_wiki_clean_surfaces
                       - oireachtas_clean_surfaces)

print(f'\n=== Phase C coverage: post-audit definitive count ===')
print(f'')
print(f'Irish Wikipedia + demutation + Wikidata:  {irish_wiki_base:>4}  ({irish_wiki_base / total_entities:.1%})')
print(f'English Wikipedia (sitelinks filter):    +{en_wiki_new:>4}  ({en_wiki_new / total_entities:.1%})')
print(f'Oireachtas API (PER, audited):           +{oireachtas_new:>4}  ({oireachtas_new / total_entities:.1%})')
print(f'Wikidata fuzzy search (audited):         +{fuzzy_new:>4}  ({fuzzy_new / total_entities:.1%})')
print(f'                                          ----')
print(f'Total covered:                            {total_covered_clean:>4}  ({coverage_rate_clean:.1%})')
print(f'Irreducible misses:                       {total_entities - total_covered_clean:>4}'
      f'  ({(total_entities - total_covered_clean) / total_entities:.1%})')

print(f'\n--- Coverage by entity type ---')
for typ in ['PER', 'ORG', 'LOC']:
    type_total   = sum(1 for s, t in entity_list if t == typ)
    type_covered = sum(1 for s, t in entity_list if t == typ and s in phase_c_covered_clean)
    print(f'  {typ}: {type_covered} / {type_total} ({type_covered / type_total:.1%})')

print(f'\n--- Empirical ceiling statement ---')
print(f'Exhaustive search across Irish Wikipedia, English Wikipedia via Wikidata')
print(f'sitelinks, GeoNames IE, Wikidata SPARQL, Oireachtas API, and Wikidata')
print(f'fuzzy search yields {coverage_rate_clean:.1%} coverage on the Adkins et al. entity')
print(f'population. The irreducible miss rate of {(total_entities - total_covered_clean)/total_entities:.1%} reflects the')
print(f'long-tail nature of the entity population in literary and journalistic')
print(f'Irish text: single-token given names, minor local organisations, and')
print(f'geographically specific place names that are absent from all public KBs.')
print(f'This ceiling is the empirical finding for the subsidiary RQ.')

Oireachtas hits before audit: 41
Rejected (false positives):   24
Clean Oireachtas hits:        17

Rejected surfaces:
  "Andrews" was -> "Barry Andrews"
  "Anthony" was -> "Richard Sidney Anthony"
  "Brendan" was -> "Brendan Corish"
  "Butler" was -> "Bernard Butler"
  "Ciarán" was -> "Ciarán Ahern"
  "Collins" was -> "Con Collins"
  "Donncha" was -> "Donnchadh Ó Briain"
  "Donohoe" was -> "Paschal Donohoe"
  "Durkan" was -> "Bernard Durkan"
  "Dónall" was -> "Dónall Ó Conalláin"
  "Fidelma" was -> "Fidelma Healy Eames"
  "Gannon" was -> "Gary Gannon"
  "Garret" was -> "Garret Ahearn"
  "Gormley" was -> "Francis Gormley"
  "John T" was -> "John Thomas Keane"
  "Jordan" was -> "Michael Jordan"
  "Kennedy" was -> "Marcella Corcoran Kennedy"
  "Lambert" was -> "C. Gordon Lambert"
  "McEntee" was -> "Helen McEntee"
  "O'Neill" was -> "Eamonn O'Neill"
  "Pádraig" was -> "Padraig Faulkner"
  "Sheehy" was -> "Timothy Sheehy"
  "Theresa" was -> "Theresa Ahearn"
  "Éamonn" was -> "Eamonn Coghl

In [28]:
# CELL 24 — unified triple set construction
#
# Assembles one TSV of (head, relation, tail) triples from all four
# coverage sources. Relation vocabulary is kept consistent with the
# Phase A Wikidata schema (instance_of, country, party, employer,
# located_in) with two additions from Oireachtas (member_of, house).
#
# Entity nodes are identified by surface string throughout (Phase C is
# surface-keyed, not QID-keyed). Tail nodes that are not themselves
# corpus entities are included as literal string nodes — PyKEEN handles
# heterogeneous node sets without special treatment.
#
# Deduplication: if a surface appears in multiple sources, triples from
# all sources are retained. Duplicate (h, r, t) triples are dropped.
#
# Input:  final_wd_keeps, en_wiki_clean, oireachtas_clean,
#         wd_fuzzy_clean, hits_exact, hits_casefold, demutation_hits,
#         org_demutation_hits, geonames_loc_lookup, entity_type_map
# Output: triples_tsv path, triple_count, entity_count, relation_count
#         written to WORKING/phase_c_triples.tsv

import csv, pickle
from collections import defaultdict

TRIPLES_PATH = f'{WORKING}/phase_c_triples.tsv'

triples = set()   # (head, relation, tail) — set deduplicates automatically

# --- Source 1: Wikidata SPARQL hits (final_wd_keeps from Cell 17) ---
# Each hit has a hit dict with relation keys from RELATIONS in Cell 5:
# instance_of, country, party, employer, located_in
WD_RELATION_KEYS = ['instance_of', 'country', 'party', 'employer', 'located_in']

for surface, typ, hit, reason in final_wd_keeps:
    for rel in WD_RELATION_KEYS:
        tail = hit.get(rel, '')
        if tail and tail.strip():
            triples.add((surface, rel, tail.strip()))

wikidata_sparql_count = len(triples)
print(f'Triples from Wikidata SPARQL (Cell 17):    {wikidata_sparql_count}')

# --- Source 2: English Wikipedia via Wikidata sitelinks (en_wiki_clean, Cell 20) ---
# en_wiki_clean: surface -> {qid, title, description, instance_of, verdict_reason}
# instance_of is a raw QID here (e.g. Q5); use description as a fallback tail
# for instance_of where the QID is not a human-readable label.
# For country and located_in, the description often encodes the country
# (e.g. "Irish politician", "British charity") — extract as a type annotation
# rather than a structured triple, since we don't have the full property set.

EN_INSTANCE_LABELS = {
    'Q5':        'human',
    'Q43229':    'organization',
    'Q2385804':  'educational_institution',
    'Q7278':     'political_party',
    'Q327333':   'government_agency',
    'Q4830453':  'business',
    'Q15401930': 'product',
    'Q11032':    'newspaper',
    'Q7366':     'song',
    'Q11424':    'film',
}

for surface, info in en_wiki_clean.items():
    # instance_of triple using readable label where available, raw QID otherwise
    p31_qid = info.get('instance_of', '')
    if p31_qid:
        tail = EN_INSTANCE_LABELS.get(p31_qid, p31_qid)
        triples.add((surface, 'instance_of', tail))

    # type triple derived from description (e.g. "Irish politician" -> NATION_OF ireland)
    # Keep this simple: add one HAS_TYPE triple using the first two words of description
    desc = info.get('description', '').strip()
    if desc:
        triples.add((surface, 'has_description_type', desc[:60]))

after_en_wiki = len(triples)
print(f'Triples from EN-Wikipedia sitelinks:       {after_en_wiki - wikidata_sparql_count}')

# --- Source 3: Oireachtas members (oireachtas_clean, Cell 23b) ---
# Relations: member_of (Dáil or Seanad), derived from URI structure.
# URI pattern: .../id/Name.D.YYYY-MM-DD  (D = Dáil, S = Seanad)

def house_from_uri(uri: str) -> str:
    if '.D.' in uri:
        return 'Dáil Éireann'
    if '.S.' in uri:
        return 'Seanad Éireann'
    return 'Oireachtas'

for surface, member in oireachtas_clean.items():
    house = house_from_uri(member.get('uri', ''))
    triples.add((surface, 'member_of', house))
    triples.add((surface, 'instance_of', 'human'))

after_oireachtas = len(triples)
print(f'Triples from Oireachtas API:               {after_oireachtas - after_en_wiki}')

# --- Source 4: Wikidata fuzzy clean (wd_fuzzy_clean, Cell 23b) ---
# {surface: {qid, label, description, verdict_reason}}
for surface, info in wd_fuzzy_clean.items():
    desc = info.get('description', '').strip()
    if desc:
        triples.add((surface, 'has_description_type', desc[:60]))
    entity_type = entity_type_map.get(surface, '')
    if entity_type == 'PER':
        triples.add((surface, 'instance_of', 'human'))
    elif entity_type == 'ORG':
        triples.add((surface, 'instance_of', 'organization'))

after_fuzzy = len(triples)
print(f'Triples from Wikidata fuzzy:               {after_fuzzy - after_oireachtas}')

# --- Source 5: GeoNames LOC entities ---
# For each covered LOC entity, add located_in Ireland and instance_of location.
# geonames_loc_lookup: surface -> GeoNames row (has 'feature_class', 'country_code')
GEONAMES_FEATURE_MAP = {
    'A': 'administrative_division',
    'P': 'populated_place',
    'H': 'water_body',
    'T': 'mountain_or_hill',
    'L': 'area',
    'R': 'road',
    'S': 'structure',
    'V': 'vegetation',
    'U': 'undersea',
}

loc_surfaces = {s for s, t in entity_list if t == 'LOC'}
geonames_covered = loc_surfaces & set(geonames_loc_lookup.keys())

for surface in geonames_covered:
    row = geonames_loc_lookup[surface]
    feature = GEONAMES_FEATURE_MAP.get(str(row.get('feature_class', '')), 'location')
    country  = str(row.get('country_code', 'IE'))
    triples.add((surface, 'instance_of', feature))
    if country == 'IE':
        triples.add((surface, 'located_in', 'Ireland'))
    else:
        triples.add((surface, 'located_in', country))

after_geonames = len(triples)
print(f'Triples from GeoNames:                     {after_geonames - after_fuzzy}')

# --- Source 6: entity type triples for all covered entities ---
# Every covered entity gets an HAS_ENTITY_TYPE triple (PER/ORG/LOC).
# This ensures every node has at least one triple, which PyKEEN requires.
TYPE_MAP = {'PER': 'person_entity', 'ORG': 'organisation_entity', 'LOC': 'location_entity'}

for surface in phase_c_covered_clean:
    entity_type = entity_type_map.get(surface, '')
    if entity_type in TYPE_MAP:
        triples.add((surface, 'has_entity_type', TYPE_MAP[entity_type]))

after_type = len(triples)
print(f'Entity type triples (coverage guarantee):  {after_type - after_geonames}')

# --- Write TSV ---
triples_list = sorted(triples)   # sort for reproducibility

with open(TRIPLES_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, delimiter='\t')
    for h, r, t in triples_list:
        writer.writerow([h, r, t])

# --- Summary ---
all_nodes     = {h for h, r, t in triples_list} | {t for h, r, t in triples_list}
all_relations = {r for h, r, t in triples_list}
head_nodes    = {h for h, r, t in triples_list}

print(f'\n=== Phase C triple set ===')
print(f'Total triples:     {len(triples_list)}')
print(f'Unique head nodes: {len(head_nodes)}  (corpus entities with at least one triple)')
print(f'Unique nodes:      {len(all_nodes)}   (entities + tail values)')
print(f'Relations:         {len(all_relations)}')
print(f'\nRelation breakdown:')
rel_counts = defaultdict(int)
for h, r, t in triples_list:
    rel_counts[r] += 1
for rel, count in sorted(rel_counts.items(), key=lambda x: -x[1]):
    print(f'  {rel:<30} {count}')
print(f'\nWritten to: {TRIPLES_PATH}')


# CELL 25 — TransE training via PyKEEN
#
# Trains a TransE model on the Phase C triple set. Hyperparameters match
# Phase A and Phase B exactly: dim=128, 100 epochs, Adam.
# Produces phase_c_embeddings.pkl: dict mapping surface string -> numpy
# array of shape (128,), covering all head nodes in the triple set.
# Uncovered entities (65.9% of corpus) receive zero vectors at lookup time.
#
# Input:  TRIPLES_PATH (from Cell 24)
# Output: WORKING/phase_c_embeddings.pkl

import subprocess
subprocess.run(['pip', 'install', 'pykeen', '-q'], check=True)

import numpy as np
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory

# --- Load triples ---
tf = TriplesFactory.from_path(TRIPLES_PATH, delimiter='\t')

print(f'TriplesFactory loaded:')
print(f'  Triples:   {tf.num_triples}')
print(f'  Entities:  {tf.num_entities}')
print(f'  Relations: {tf.num_relations}')

# Three-way split: PyKEEN pipeline requires training, validation, and testing.
tf_train, tf_val, tf_test = tf.split([0.8, 0.1, 0.1], random_state=42)

print(f'\nSplit:')
print(f'  Train: {tf_train.num_triples}')
print(f'  Val:   {tf_val.num_triples}')
print(f'  Test:  {tf_test.num_triples}')

# --- Train ---
print('\nTraining TransE (dim=128, 100 epochs)...')

result = pipeline(
    training=tf_train,
    validation=tf_val,
    testing=tf_test,
    model='TransE',
    model_kwargs=dict(
        embedding_dim=KG_DIM,
    ),
    optimizer='Adam',
    optimizer_kwargs=dict(lr=0.01),
    training_kwargs=dict(
        num_epochs=100,
        batch_size=256,
    ),
    random_seed=42,
    device=device,
    use_tqdm=True,
)

print(f'\nFinal loss: {result.losses[-1]:.4f}')

# --- Extract embeddings ---
# pykeen 1.10+ stores entity embeddings at result.model.entity_representations[0]
entity_repr  = result.model.entity_representations[0]
embed_matrix = entity_repr(indices=None).detach().cpu().numpy()   # (num_entities, 128)

# Build surface -> embedding dict using the TriplesFactory entity-to-id map
entity_to_id  = tf.entity_to_id   # dict: entity_label -> int index
id_to_entity  = {v: k for k, v in entity_to_id.items()}

surface_to_embedding = {}
for entity_label, idx in entity_to_id.items():
    surface_to_embedding[entity_label] = embed_matrix[idx]

print(f'\nEmbedding dict size: {len(surface_to_embedding)} entities')
print(f'Embedding dim:       {embed_matrix.shape[1]}')

# --- Sanity check: sample a few corpus entity lookups ---
sample_surfaces = [s for s in list(phase_c_covered_clean)[:5]]
print(f'\nSanity check (5 covered entities):')
for s in sample_surfaces:
    vec = surface_to_embedding.get(s)
    if vec is not None:
        print(f'  "{s[:40]:<40}" norm={np.linalg.norm(vec):.4f}')
    else:
        print(f'  "{s[:40]:<40}" MISSING — not in triple set')

# --- Save ---
EMBEDDINGS_PATH = f'{WORKING}/phase_c_embeddings.pkl'
with open(EMBEDDINGS_PATH, 'wb') as f:
    pickle.dump(surface_to_embedding, f)

print(f'\nSaved to: {EMBEDDINGS_PATH}')
print(f'File size: {__import__("os").path.getsize(EMBEDDINGS_PATH) / 1024:.1f} KB')

# --- Coverage check: what fraction of corpus entities are in the dict? ---
all_corpus_surfaces = {s for s, t in entity_list}
in_dict  = all_corpus_surfaces & set(surface_to_embedding.keys())
out_dict = all_corpus_surfaces - set(surface_to_embedding.keys())

print(f'\nCorpus entity coverage in embedding dict:')
print(f'  In dict:  {len(in_dict)} / {len(all_corpus_surfaces)} ({len(in_dict)/len(all_corpus_surfaces):.1%})')
print(f'  Out dict: {len(out_dict)} ({len(out_dict)/len(all_corpus_surfaces):.1%}) — zero vector at NER lookup')

Triples from Wikidata SPARQL (Cell 17):    29
Triples from EN-Wikipedia sitelinks:       143
Triples from Oireachtas API:               34
Triples from Wikidata fuzzy:               24
Triples from GeoNames:                     102
Entity type triples (coverage guarantee):  646

=== Phase C triple set ===
Total triples:     978
Unique head nodes: 649  (corpus entities with at least one triple)
Unique nodes:      770   (entities + tail values)
Relations:         5

Relation breakdown:
  has_entity_type                646
  instance_of                    180
  has_description_type           84
  located_in                     51
  member_of                      17

Written to: /kaggle/working/phase_c_triples.tsv
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training epochs on cpu:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cpu:   0%|          | 0.00/4.00 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/4.00 [00:00<?, ?batch/s]

...

Training batches on cpu:   0%|          | 0.00/4.00 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/4.00 [00:00<?, ?batch/s]

Evaluating on cpu:   0%|          | 0.00/98.0 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 0.15s seconds



Final loss: 0.1436

Embedding dict size: 770 entities
Embedding dim:       128

Sanity check (5 covered entities):
  "Máirtín Ó Cadhain                       " norm=1.0000
  "Ombudsman Eorpach                       " norm=1.0000
  "Comhairle Contae na Mí                  " norm=1.0000
  "Brandenburg                             " norm=1.0000
  "Brian Ó Catháin                         " norm=1.0000

Saved to: /kaggle/working/phase_c_embeddings.pkl
File size: 423.7 KB

Corpus entity coverage in embedding dict:
  In dict:  649 / 1863 (34.8%)
  Out dict: 1214 (65.2%) — zero vector at NER lookup


In [29]:
# CELL 26a — tokenizer, label vocabulary, and test split loader

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)

train_tokens, train_labels_raw = load_conll(f'{DATA}/conll/train_final.conll')
all_labels = sorted(set(l for seq in train_labels_raw for l in seq))
label2id   = {l: i for i, l in enumerate(all_labels)}
id2label   = {i: l for l, i in label2id.items()}

test_tokens, test_labels = load_conll(f'{DATA}/conll/NER_Irish_test.conll')

print(f'Label vocabulary: {all_labels}')
print(f'Train sentences:  {len(train_tokens)}')
print(f'Test sentences:   {len(test_tokens)}')

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Label vocabulary: ['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']
Train sentences:  1006
Test sentences:   140


In [30]:
# CELL 26b — GaBERTCRF architecture and evaluation function

class GaBERTCRF(nn.Module):
    def __init__(self, model_name, num_labels, fusion_type='late', kg_dim=128):
        super().__init__()
        self.fusion_type = fusion_type
        self.num_labels  = num_labels
        self.kg_dim      = kg_dim
        self.bert = AutoModel.from_pretrained(
            model_name,
            output_hidden_states=True,
        )
        if fusion_type == 'late':
            classifier_input_dim = 4 * BERT_DIM + kg_dim
        elif fusion_type == 'additive':
            classifier_input_dim = BERT_DIM
            self.kg_projection = nn.Linear(kg_dim, BERT_DIM)
        else:
            raise ValueError(f"fusion_type must be 'late' or 'additive', got '{fusion_type}'")
        self.classifier = nn.Linear(classifier_input_dim, num_labels)
        self.crf        = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, kg_vectors, labels=None):
        outputs       = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.hidden_states
        if self.fusion_type == 'late':
            bert_repr = torch.cat(hidden_states[-4:], dim=-1)
            combined  = torch.cat([bert_repr, kg_vectors], dim=-1)
        elif self.fusion_type == 'additive':
            bert_repr    = hidden_states[-1]
            kg_projected = self.kg_projection(kg_vectors)
            combined     = bert_repr + kg_projected
        emissions = self.classifier(combined)
        if labels is not None:
            mask       = (labels != -100)
            mask[:, 0] = True
            labels_crf = labels.clone()
            labels_crf[~mask] = 0
            labels_crf[:, 0] = 0   # CLS token label set to 0; position excluded from seqeval via -100 in label_ids
            loss = -self.crf(emissions, labels_crf, mask=mask, reduction='mean')
        else:
            mask = (attention_mask == 1)
            mask[:, 0] = True   # CRF requires first timestep unmasked
            preds = self.crf.decode(emissions, mask=mask)
            return preds


def evaluate_subset(model, input_ids, attention_mask, label_ids, kg_vectors, id2label):
    model.eval()
    with torch.no_grad():
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        kg_vectors     = kg_vectors.to(device)
        label_ids      = label_ids.to(device)
        preds = model(input_ids, attention_mask, kg_vectors)
    true_labels, pred_labels = [], []
    for i, pred_seq in enumerate(preds):
        true_seq = label_ids[i].cpu().numpy()
        true_row, pred_row = [], []
        for j, pred_id in enumerate(pred_seq):
            if true_seq[j] == -100:
                continue
            true_row.append(id2label[true_seq[j]])
            pred_row.append(id2label[pred_id])
        true_labels.append(true_row)
        pred_labels.append(pred_row)
    return f1_score(true_labels, pred_labels)

print('GaBERTCRF and evaluate_subset defined')

GaBERTCRF and evaluate_subset defined


In [31]:
# CELL 26c — Phase C NER ablation runs (With Fast Recovery Bypass)

import json, random, os, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from torchcrf import CRF
from seqeval.metrics import f1_score as seqeval_f1

# --- Automated Recovery / Skip Check ---
DATASET_BASE = '/kaggle/input/gabert-evaluation-assets'
SAVED_RESULTS_PATH = f'{DATASET_BASE}/working/phase_c_all_results.json'
TARGET_RESULTS_PATH = f'{WORKING}/phase_c_all_results.json'

if os.path.exists(SAVED_RESULTS_PATH):
    print("=== [BYPASS ACTIVE] Pre-computed Phase C metrics found! ===")
    os.makedirs(WORKING, exist_ok=True)
    shutil.copy(SAVED_RESULTS_PATH, TARGET_RESULTS_PATH)
    print(f"-> Restored pre-computed matrix to: {TARGET_RESULTS_PATH}")

    with open(TARGET_RESULTS_PATH, 'r') as f:
        all_results = json.load(f)

    phase_c_results    = all_results.get('phase_c_full_and_covered', {})
    prior_on_c_covered = all_results.get('prior_conditions_on_c_covered', {})

    # Diagnostic: confirm seed keys match SEEDS
    sample_condition = list(phase_c_results.keys())[0]
    stored_keys = list(phase_c_results[sample_condition].keys())
    print(f"\nStored seed keys ({sample_condition}): {stored_keys}")
    expected_keys = [str(s) for s in SEEDS]
    if sorted(stored_keys) != sorted(expected_keys):
        print(f"WARNING: stored keys {stored_keys} do not match SEEDS {SEEDS} — check Cell 2")
    else:
        print("Seed keys match SEEDS list. OK.")

    # --- Summary table ---
    print('\n=== Phase C results summary ===')
    print(f'\n{"Condition":<14} {"Mean F1 (full)":>16} {"Mean F1 (covered)":>19}')
    print('-' * 52)
    for condition in ['C_late', 'C_additive']:
        if condition in phase_c_results:
            seed_results   = phase_c_results[condition]
            full_scores    = [v['full']    for v in seed_results.values()]
            covered_scores = [v['covered'] for v in seed_results.values()]
            print(f'{condition:<14} {np.mean(full_scores):>16.4f} {np.mean(covered_scores):>19.4f}')

    print(f'\n{"Condition":<14} {"Mean F1 (C-covered subset)":>28}')
    print('-' * 44)
    for condition in ['no_kg', 'A_late', 'A_additive', 'B_late', 'B_additive']:
        if condition in prior_on_c_covered:
            scores = list(prior_on_c_covered[condition].values())
            print(f'{condition:<14} {np.mean(scores):>28.4f}')

else:
    print("=== [FALLBACK] Pre-computed metrics not found. Running full training loop... ===")
    # (fallback training code unchanged — omitted here for brevity,
    #  keep everything from the original cell from this point down)

=== [FALLBACK] Pre-computed metrics not found. Running full training loop... ===


In [34]:
# CELL 26c — Phase C NER ablation runs (With Fast Recovery Bypass)
#
# Trains and evaluates two Phase C conditions (C_late, C_additive) across
# 7 seeds each, using the phase_c_embeddings.pkl produced in Cell 25.
#
# MODIFICATION: Bypasses execution entirely if recovery metrics are found
# inside the attached Kaggle dataset assets folder.

import json, random, os, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from torchcrf import CRF
from seqeval.metrics import f1_score as seqeval_f1

# --- Automated Recovery / Skip Check ---
DATASET_BASE = '/kaggle/input/datasets/michaelmarkey64/gabert-evaluation-assets'
SAVED_RESULTS_PATH = f'{DATASET_BASE}/working/phase_c_all_results.json'
TARGET_RESULTS_PATH = f'{WORKING}/phase_c_all_results.json'

if os.path.exists(SAVED_RESULTS_PATH):
    print("=== [BYPASS ACTIVE] Pre-computed Phase C metrics found! ===")
    os.makedirs(WORKING, exist_ok=True)
    shutil.copy(SAVED_RESULTS_PATH, TARGET_RESULTS_PATH)
    print(f"-> Restored pre-computed matrix to: {TARGET_RESULTS_PATH}")
    
    with open(TARGET_RESULTS_PATH, 'r') as f:
        all_results = json.load(f)
        
    phase_c_results = all_results.get('phase_c_full_and_covered', {})
    prior_on_c_covered = all_results.get('prior_conditions_on_c_covered', {})

    # --- Summary table ---
    print('\n=== Phase C results summary ===')
    print(f'\n{"Condition":<14} {"Mean F1 (full)":>16} {"Mean F1 (covered)":>19}')
    print('-' * 52)
    for condition in ['C_late', 'C_additive']:
        if condition in phase_c_results:
            seed_results = phase_c_results[condition]
            full_scores    = [v['full']    for v in seed_results.values()]
            covered_scores = [v['covered'] for v in seed_results.values()]
            print(f'{condition:<14} {np.mean(full_scores):>16.4f} {np.mean(covered_scores):>19.4f}')

    print(f'\n{"Condition":<14} {"Mean F1 (C-covered subset)":>28}')
    print('-' * 44)
    for condition in ['no_kg', 'A_late', 'A_additive', 'B_late', 'B_additive']:
        if condition in prior_on_c_covered:
            scores = list(prior_on_c_covered[condition].values())
            print(f'{condition:<14} {np.mean(scores):>28.4f}')
else:
    print("=== [FALLBACK] Pre-computed metrics not found. Running full training loop... ===")
    
    # --- Load Phase C embeddings ---
    with open(EMBEDDINGS_PATH, 'rb') as f:
        emb_c = pickle.load(f)

    print(f'Phase C embeddings loaded: {len(emb_c)} entities')

    # --- get_kg_vector for Phase C (surface-keyed, no QID indirection) ---
    def get_kg_vector_c(surface, emb_c, dim):
        vec = emb_c.get(surface)
        if vec is None:
            return np.zeros(dim, dtype=np.float32)
        return vec.astype(np.float32)

    # --- Encode test split with Phase C KG vectors ---
    def encode_split_with_kg(sentences, labels, tokenizer, label2id,
                              kg_lookup_fn, max_len=128):
        all_input_ids, all_attention_mask, all_label_ids, all_kg = [], [], [], []

        for tokens, token_labels in zip(sentences, labels):
            enc = tokenizer(
                tokens,
                is_split_into_words=True,
                max_length=max_len,
                padding='max_length',
                truncation=True,
                return_tensors='pt',
            )
            input_ids      = enc['input_ids'].squeeze(0)
            attention_mask = enc['attention_mask'].squeeze(0)
            word_ids       = enc.word_ids(batch_index=0)

            label_ids  = []
            kg_vectors = []
            prev_word  = None

            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                    kg_vectors.append(np.zeros(KG_DIM, dtype=np.float32))
                elif word_idx != prev_word:
                    label_ids.append(label2id[token_labels[word_idx]])
                    kg_vectors.append(kg_lookup_fn(tokens[word_idx]))
                    prev_word = word_idx
                else:
                    label_ids.append(-100)
                    kg_vectors.append(np.zeros(KG_DIM, dtype=np.float32))

            all_input_ids.append(input_ids)
            all_attention_mask.append(attention_mask)
            all_label_ids.append(torch.tensor(label_ids, dtype=torch.long))
            all_kg.append(torch.tensor(np.stack(kg_vectors), dtype=torch.float32))

        return (
            torch.stack(all_input_ids),
            torch.stack(all_attention_mask),
            torch.stack(all_label_ids),
            torch.stack(all_kg),
        )

    # Encode training split with Phase C vectors
    print('Encoding training split with Phase C KG vectors...')
    train_tokens, train_labels_seq = load_conll(f'{DATA}/conll/train_final.conll')
    train_ids_c, train_mask_c, train_labs_c, train_kg_c = encode_split_with_kg(
        train_tokens, train_labels_seq, tokenizer, label2id,
        lambda s: get_kg_vector_c(s, emb_c, KG_DIM),
    )
    print(f'  Train sentences encoded: {len(train_ids_c)}')

    # Encode test split with Phase C vectors
    print('Encoding test split with Phase C KG vectors...')
    test_ids_c, test_mask_c, test_labs_c, test_kg_c = encode_split_with_kg(
        test_tokens, test_labels, tokenizer, label2id,
        lambda s: get_kg_vector_c(s, emb_c, KG_DIM),
    )
    print(f'  Test sentences encoded:  {len(test_ids_c)}')

    # Identify Phase C covered subset of test sentences
    covered_c_idx = []
    for i in range(len(test_ids_c)):
        kg_row = test_kg_c[i]                          # (max_len, 128)
        norms  = kg_row.norm(dim=-1)                   # (max_len,)
        if (norms > 0).any():
            covered_c_idx.append(i)

    print(f'\nPhase C covered test sentences: {len(covered_c_idx)} / {len(test_ids_c)}')

    # Subset tensors for covered sentences
    c_idx_t = torch.tensor(covered_c_idx)
    test_ids_c_sub   = test_ids_c[c_idx_t]
    test_mask_c_sub  = test_mask_c[c_idx_t]
    test_labs_c_sub  = test_labs_c[c_idx_t]
    test_kg_c_sub    = test_kg_c[c_idx_t]

    # --- Training function ---
    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    def train_condition(fusion_type, seed,
                        train_ids, train_mask, train_labs, train_kg):
        set_seed(seed)
        model = GaBERTCRF(
            model_name=MODEL_NAME,
            num_labels=len(label2id),
            fusion_type=fusion_type,
            kg_dim=KG_DIM,
        ).to(device)

        dataset    = TensorDataset(train_ids, train_mask, train_labs, train_kg)
        loader     = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
        optimizer  = optim.AdamW(model.parameters(), lr=LR)

        model.train()
        
        n_all_masked = sum(1 for i in range(len(train_labs_c)) if (train_labs_c[i] != -100).sum() == 0)
        print(f'Fully masked training sentences: {n_all_masked} / {len(train_labs_c)}')
        
        for epoch in range(EPOCHS):
            epoch_loss = 0.0
            for batch in loader:
                b_ids, b_mask, b_labs, b_kg = [x.to(device) for x in batch]
        
                # skip batches with no real labels
                if (b_labs != -100).sum() == 0:
                    continue
        
                optimizer.zero_grad()
                loss = model(b_ids, b_mask, b_kg, labels=b_labs)
                if loss is None:
                    continue
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                epoch_loss += loss.item()
            avg = epoch_loss / len(loader)
            if (epoch + 1) % 5 == 0:
                print(f'    epoch {epoch+1}/{EPOCHS}  loss={avg:.4f}')

        return model

    # --- Run Phase C conditions ---
    C_CONDITIONS = {
        'C_late':     'late',
        'C_additive': 'additive',
    }

    phase_c_results = {}

    for condition, fusion_type in C_CONDITIONS.items():
        phase_c_results[condition] = {}
        print(f'\n--- {condition} ---')

        for seed in SEEDS:
            ckpt_path = f'{WORKING}/{condition}_seed{seed}.pt'
            print(f'  seed {seed}:')

            model = train_condition(
                fusion_type, seed,
                train_ids_c, train_mask_c, train_labs_c, train_kg_c,
            )

            torch.save(model.state_dict(), ckpt_path)

            # Full test set F1
            f1_full = evaluate_subset(
                model, test_ids_c, test_mask_c, test_labs_c, test_kg_c, id2label,
            )
            # Covered subset F1
            f1_covered = evaluate_subset(
                model, test_ids_c_sub, test_mask_c_sub,
                test_labs_c_sub, test_kg_c_sub, id2label,
            )

            phase_c_results[condition][seed] = {
                'full':    round(f1_full, 4),
                'covered': round(f1_covered, 4),
            }
            print(f'    F1 full={f1_full:.4f}  covered={f1_covered:.4f}')

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print('\nPhase C training complete.')

    # --- Re-evaluate existing conditions on Phase C covered subset ---
    print('\n--- Re-evaluating A/B/no_kg conditions on Phase C covered subset ---')

    with open(f'{DATA}/phase_a_embeddings.pkl', 'rb') as f:
        emb_a = pickle.load(f)
    with open(f'{DATA}/phase_a_surface_to_qid.pkl', 'rb') as f:
        surface_to_qid = pickle.load(f)
    with open(f'{DATA}/phase_b_embeddings.pkl', 'rb') as f:
        emb_b = pickle.load(f)

    def encode_kg_for_condition(condition, sentences):
        if condition == 'no_kg':
            return torch.zeros(len(test_ids_c), MAX_LEN, KG_DIM, dtype=torch.float32)
        if condition in ('A_late', 'A_additive'):
            lookup = lambda s: get_kg_vector_a(s, emb_a, surface_to_qid, KG_DIM)
        else:
            lookup = lambda s: get_kg_vector_b(s, emb_b, KG_DIM)

        _, _, _, kg = encode_split_with_kg(test_tokens, test_labels, tokenizer, label2id, lookup)
        return kg

    PRIOR_CONDITIONS = {
        'no_kg':      'late',
        'A_late':     'late',
        'A_additive': 'additive',
        'B_late':     'late',
        'B_additive': 'additive',
    }

    prior_on_c_covered = {}

    for condition, fusion_type in PRIOR_CONDITIONS.items():
        prior_on_c_covered[condition] = {}
        print(f'\n  {condition}:')
        kg_full = encode_kg_for_condition(condition, test_tokens)
        kg_sub  = kg_full[c_idx_t]

        for seed in SEEDS:
            ckpt_path = f'{CHECKPOINTS}/{condition}_seed{seed}.pt'
            model = GaBERTCRF(
                model_name=MODEL_NAME,
                num_labels=len(label2id),
                fusion_type=fusion_type,
                kg_dim=KG_DIM,
            ).to(device)
            model.load_state_dict(torch.load(ckpt_path, map_location=device))

            f1_covered = evaluate_subset(
                model, test_ids_c_sub, test_mask_c_sub, test_labs_c_sub, kg_sub, id2label
            )
            prior_on_c_covered[condition][seed] = round(f1_covered, 4)
            print(f'    seed {seed}: F1 covered = {f1_covered:.4f}')

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # --- Save all results ---
    all_results = {
        'phase_c_full_and_covered': phase_c_results,
        'prior_conditions_on_c_covered': prior_on_c_covered,
    }

    RESULTS_PATH = f'{WORKING}/phase_c_all_results.json'
    with open(RESULTS_PATH, 'w') as f:
        json.dump(all_results, f, indent=2)

    print(f'\nResults saved to: {RESULTS_PATH}')

    # --- Summary table ---
    print('\n=== Phase C results summary ===')
    print(f'\n{"Condition":<14} {"Mean F1 (full)":>16} {"Mean F1 (covered)":>19}')
    print('-' * 52)
    for condition in C_CONDITIONS:
        seed_results = phase_c_results[condition]
        full_scores    = [v['full']    for v in seed_results.values()]
        covered_scores = [v['covered'] for v in seed_results.values()]
        print(f'{condition:<14} {np.mean(full_scores):>16.4f} {np.mean(covered_scores):>19.4f}')

    print(f'\n{"Condition":<14} {"Mean F1 (C-covered subset)":>28}')
    print('-' * 44)
    for condition in PRIOR_CONDITIONS:
        scores = list(prior_on_c_covered[condition].values())
        print(f'{condition:<14} {np.mean(scores):>28.4f}')

=== [BYPASS ACTIVE] Pre-computed Phase C metrics found! ===
-> Restored pre-computed matrix to: /kaggle/working/phase_c_all_results.json

=== Phase C results summary ===

Condition        Mean F1 (full)   Mean F1 (covered)
----------------------------------------------------
C_late                   0.0180              0.0240
C_additive               0.0163              0.0218

Condition        Mean F1 (C-covered subset)
--------------------------------------------
no_kg                                0.7377
A_late                               0.0000
A_additive                           0.0000
B_late                               0.7306
B_additive                           0.7296


In [38]:
# CELL 27 — statistical tests and results table
# FIXED:
#   (1) extract_seed_scores now reads 'test_f1' from ablation_results.json
#       dicts (val_f1/test_f1 schema) rather than falling back to list(val.values())[0]
#       which was returning val_f1 and inflating the no_kg baseline to ~0.83.
#   (2) All-zero A_late/A_additive in prior_conditions_on_c_covered are a data
#       integrity issue in the stored asset — flagged explicitly below.
#       The Phase C scores (~0.018) are also from a broken training run — see note.

from scipy.stats import wilcoxon
import numpy as np
import json
import os

# --- Paths (SEEDS inherited from Cell 2) ---
WORKING      = '/kaggle/working'
DATASET_BASE = '/kaggle/input/datasets/michaelmarkey64/gabert-evaluation-assets/working'
ABLATION     = DATASET_BASE if not os.path.exists(f'{WORKING}/ablation_results.json') else WORKING

# --- Load Phase C results ---
with open(f'{WORKING}/phase_c_all_results.json') as f:
    all_results = json.load(f)

phase_c       = all_results['phase_c_full_and_covered']
prior_covered = all_results['prior_conditions_on_c_covered']

# --- Diagnostic: confirm seed keys ---
sample_condition = list(phase_c.keys())[0]
stored_keys      = list(phase_c[sample_condition].keys())
print(f"Seed keys in JSON ({sample_condition}): {stored_keys}")
expected_keys = [str(s) for s in SEEDS]
if sorted(stored_keys) != sorted(expected_keys):
    print(f"WARNING: mismatch — stored {stored_keys} vs SEEDS {SEEDS}")
else:
    print("Seed keys match SEEDS. OK.")

# --- Load prior A/B/no_kg full-test results ---
with open(f'{ABLATION}/ablation_results.json') as f:
    prior_full = json.load(f)

# --- FIX 1: extract_numeric_value now handles val_f1/test_f1 dicts correctly ---
# ablation_results.json stores per-seed results as {"val_f1": X, "test_f1": Y}.
# We always want test_f1 for the held-out evaluation. The old fallback to
# list(val.values())[0] was returning val_f1 instead, inflating no_kg to ~0.83.

def extract_numeric_value(val):
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, dict):
        # Prefer test_f1 (ablation_results.json schema)
        if 'test_f1' in val:
            return float(val['test_f1'])
        # Phase C schema uses 'full' / 'covered'
        for k in ['full', 'f1', 'macro_f1', 'value']:
            if k in val:
                return extract_numeric_value(val[k])
        # Last resort: first value (should not be reached with above guards)
        if val:
            return extract_numeric_value(list(val.values())[0])
    if isinstance(val, list) and val:
        return extract_numeric_value(val[0])
    return 0.0


def extract_seed_scores(data_dict, condition, seeds_list):
    cond_data = data_dict.get(condition, [])
    if isinstance(cond_data, list):
        if len(cond_data) >= len(seeds_list):
            return [extract_numeric_value(x) for x in cond_data[:len(seeds_list)]]
        return [extract_numeric_value(x) for x in cond_data] + [0.0] * (len(seeds_list) - len(cond_data))
    if isinstance(cond_data, dict):
        scores = []
        for s in seeds_list:
            val = None
            if str(s) in cond_data:
                val = cond_data[str(s)]
            elif s in cond_data:
                val = cond_data[s]
            elif f"seed{s}" in cond_data:
                val = cond_data[f"seed{s}"]
            else:
                keys = list(cond_data.keys())
                val  = cond_data[keys[s - 1]] if keys and (s - 1) < len(keys) else None
            scores.append(extract_numeric_value(val))
        return scores
    return [0.0] * len(seeds_list)


def extract_phase_c_scores(phase_c_dict, condition, seeds_list, metric_key):
    cond_data = phase_c_dict.get(condition, {})
    scores = []
    for s in seeds_list:
        seed_data = cond_data.get(s) or cond_data.get(str(s)) or cond_data.get(f"seed{s}")
        if isinstance(seed_data, dict) and metric_key in seed_data:
            scores.append(extract_numeric_value(seed_data[metric_key]))
        else:
            scores.append(extract_numeric_value(seed_data))
    return scores


# --- Assemble F1 vectors ---
full_f1, covered_f1 = {}, {}

for condition in ['no_kg', 'A_late', 'A_additive', 'B_late', 'B_additive']:
    full_f1[condition]    = extract_seed_scores(prior_full,    condition, SEEDS)
    covered_f1[condition] = extract_seed_scores(prior_covered, condition, SEEDS)

for condition in ['C_late', 'C_additive']:
    full_f1[condition]    = extract_phase_c_scores(phase_c, condition, SEEDS, 'full')
    covered_f1[condition] = extract_phase_c_scores(phase_c, condition, SEEDS, 'covered')

# Baseline derived from saved no_kg test_f1 scores (not hardcoded)
BASELINE_F1 = np.mean(full_f1['no_kg'])
print(f'\nDerived no_kg baseline F1 (test_f1): {BASELINE_F1:.4f}')
print(f'  Per-seed: {[round(x,4) for x in full_f1["no_kg"]]}')

# --- FIX 2: Data integrity warnings ---
# Flag conditions with all-zero or near-zero scores so downstream
# interpretation is not silently distorted.

print('\n--- Data integrity check ---')
for condition in ['A_late', 'A_additive']:
    scores = covered_f1[condition]
    if all(s == 0.0 for s in scores):
        print(f'WARNING: {condition} covered scores are all 0.0 in prior_conditions_on_c_covered.')
        print(f'  This means the stored phase_c_all_results.json was produced by a run that')
        print(f'  did not successfully re-evaluate A conditions on the Phase C covered subset.')
        print(f'  These conditions CANNOT be compared on the covered subset with current data.')
        print(f'  Options: (a) rerun Cell 26c in fallback mode to regenerate the stored JSON;')
        print(f'           (b) omit A covered-subset comparisons from the dissertation table.')

for condition in ['C_late', 'C_additive']:
    scores = full_f1[condition]
    mean_c = np.mean(scores)
    if mean_c < 0.1:
        print(f'WARNING: {condition} full-test mean F1 = {mean_c:.4f} — near-zero, not a null result.')
        print(f'  The stored phase_c_all_results.json contains broken training outputs.')
        print(f'  The Phase C training run (Cell 26c fallback) must be rerun to get valid scores.')
        print(f'  Do NOT use these scores for dissertation reporting.')

print()

ALL_CONDITIONS = ['no_kg', 'A_late', 'A_additive', 'B_late', 'B_additive', 'C_late', 'C_additive']

# --- Full test set results table ---
print('=== Full test set F1 (7 seeds) ===')
print(f'{"Condition":<14} {"Mean":>7} {"SD":>7} {"Min":>7} {"Max":>7}')
print('-' * 44)
for c in ALL_CONDITIONS:
    scores = full_f1[c]
    note = '  *** BROKEN — see warning above ***' if np.mean(scores) < 0.1 else ''
    print(f'{c:<14} {np.mean(scores):>7.4f} {np.std(scores):>7.4f} '
          f'{np.min(scores):>7.4f} {np.max(scores):>7.4f}{note}')

# --- Covered subset results table ---
print(f'\n=== Covered subset F1 (Phase C sentences, 7 seeds) ===')
print(f'{"Condition":<14} {"Mean":>7} {"SD":>7} {"Min":>7} {"Max":>7}')
print('-' * 44)
for c in ALL_CONDITIONS:
    scores = covered_f1[c]
    note = '  *** zero — missing data ***' if all(s == 0.0 for s in scores) else ''
    note = '  *** BROKEN ***' if np.mean(scores) < 0.1 and c in ['C_late','C_additive'] else note
    print(f'{c:<14} {np.mean(scores):>7.4f} {np.std(scores):>7.4f} '
          f'{np.min(scores):>7.4f} {np.max(scores):>7.4f}{note}')

# --- One-sample Wilcoxon vs no_kg baseline ---
# Only run on conditions with valid (non-broken) scores.
VALID_CONDITIONS = [c for c in ALL_CONDITIONS
                    if np.mean(full_f1[c]) > 0.1]

print(f'\n=== One-sample Wilcoxon vs no_kg baseline (F1={BASELINE_F1:.4f}) ===')
print(f'  (skipping conditions with broken/zero scores)')
print(f'{"Condition":<14} {"Mean F1":>9} {"Diff":>7} {"stat":>8} {"p":>8}')
print('-' * 50)
for c in VALID_CONDITIONS:
    scores = full_f1[c]
    diffs  = [s - BASELINE_F1 for s in scores]
    if all(d == 0 for d in diffs) or np.std(diffs) == 0:
        print(f'{c:<14} {np.mean(scores):>9.4f} {np.mean(diffs):>+7.4f}   {"N/A":>8} {"N/A":>8}')
        continue
    stat, p = wilcoxon(diffs, alternative='two-sided')
    print(f'{c:<14} {np.mean(scores):>9.4f} {np.mean(diffs):>+7.4f} {stat:>8.1f} {p:>8.4f}')

# --- Two-sample Wilcoxon: C conditions vs no_kg (only if C scores are valid) ---
c_valid = np.mean(full_f1['C_late']) > 0.1

print(f'\n=== Two-sample Wilcoxon: C conditions vs no_kg (full test set) ===')
if not c_valid:
    print('  SKIPPED — Phase C scores are from a broken run. Rerun Cell 26c first.')
else:
    for c in ['C_late', 'C_additive']:
        diff_arr = np.array(full_f1[c]) - np.array(full_f1['no_kg'])
        diff     = np.mean(diff_arr)
        if np.all(diff_arr == 0) or np.std(diff_arr) == 0:
            print(f'{c} vs no_kg:  diff={diff:+.4f}  stat=N/A  p=N/A (zero variance)')
            continue
        stat, p = wilcoxon(full_f1[c], full_f1['no_kg'], alternative='two-sided')
        print(f'{c} vs no_kg:  diff={diff:+.4f}  stat={stat:.1f}  p={p:.4f}')

# --- Two-sample Wilcoxon: C vs A and C vs B ---
print(f'\n=== Two-sample Wilcoxon: C vs A and C vs B (full test set) ===')
if not c_valid:
    print('  SKIPPED — Phase C scores are from a broken run.')
else:
    for c1, c2 in [('C_late', 'A_late'), ('C_late', 'B_late'),
                   ('C_additive', 'A_additive'), ('C_additive', 'B_additive')]:
        diff_arr = np.array(full_f1[c1]) - np.array(full_f1[c2])
        diff     = np.mean(diff_arr)
        if np.all(diff_arr == 0) or np.std(diff_arr) == 0:
            print(f'{c1} vs {c2}:  diff={diff:+.4f}  stat=N/A  p=N/A (zero variance)')
            continue
        stat, p = wilcoxon(full_f1[c1], full_f1[c2], alternative='two-sided')
        print(f'{c1} vs {c2}:  diff={diff:+.4f}  stat={stat:.1f}  p={p:.4f}')

# --- Two-sample Wilcoxon: C vs no_kg on covered subset ---
print(f'\n=== Two-sample Wilcoxon: C vs no_kg (covered subset) ===')
if not c_valid:
    print('  SKIPPED — Phase C scores are from a broken run.')
else:
    for c in ['C_late', 'C_additive']:
        diff_arr = np.array(covered_f1[c]) - np.array(covered_f1['no_kg'])
        diff     = np.mean(diff_arr)
        if np.all(diff_arr == 0) or np.std(diff_arr) == 0:
            print(f'{c} vs no_kg (covered):  diff={diff:+.4f}  stat=N/A  p=N/A (zero variance)')
            continue
        stat, p = wilcoxon(covered_f1[c], covered_f1['no_kg'], alternative='two-sided')
        print(f'{c} vs no_kg (covered):  diff={diff:+.4f}  stat={stat:.1f}  p={p:.4f}')

# --- A/B vs no_kg on Phase C covered subset (B only — A data is missing) ---
print(f'\n=== Two-sample Wilcoxon: B conditions vs no_kg (Phase C covered subset) ===')
print('  (A conditions excluded — all-zero scores indicate missing data in stored JSON)')
for c in ['B_late', 'B_additive']:
    scores_c = covered_f1[c]
    scores_baseline = covered_f1['no_kg']
    if all(s == 0.0 for s in scores_c) or all(s == 0.0 for s in scores_baseline):
        print(f'{c} vs no_kg (covered):  SKIPPED — zero scores')
        continue
    diff_arr = np.array(scores_c) - np.array(scores_baseline)
    diff     = np.mean(diff_arr)
    if np.all(diff_arr == 0) or np.std(diff_arr) == 0:
        print(f'{c} vs no_kg (covered):  diff={diff:+.4f}  stat=N/A  p=N/A (zero variance)')
        continue
    stat, p = wilcoxon(scores_c, scores_baseline, alternative='two-sided')
    print(f'{c} vs no_kg (covered):  diff={diff:+.4f}  stat={stat:.1f}  p={p:.4f}')

Seed keys in JSON (C_late): ['42', '123', '256', '512', '999', '1024', '2048']
Seed keys match SEEDS. OK.

Derived no_kg baseline F1: 0.8312

=== Full test set F1 (7 seeds) ===
Condition         Mean      SD     Min     Max
--------------------------------------------
no_kg           0.8312  0.0088  0.8148  0.8394
A_late          0.8306  0.0082  0.8148  0.8393
A_additive      0.8296  0.0087  0.8123  0.8419
B_late          0.8297  0.0089  0.8144  0.8382
B_additive      0.8287  0.0077  0.8141  0.8404
C_late          0.0180  0.0053  0.0111  0.0274
C_additive      0.0163  0.0043  0.0106  0.0252

=== Covered subset F1 (Phase C sentences, 7 seeds) ===
Condition         Mean      SD     Min     Max
--------------------------------------------
no_kg           0.7377  0.0144  0.7133  0.7588
A_late          0.0000  0.0000  0.0000  0.0000
A_additive      0.0000  0.0000  0.0000  0.0000
B_late          0.7306  0.0163  0.7043  0.7606
B_additive      0.7296  0.0059  0.7195  0.7406
C_late          0.0

In [39]:
import json
import os

# 1. Check which ablation_results.json is being used
WORKING = '/kaggle/working'
DATASET_BASE = '/kaggle/input/datasets/michaelmarkey64/gabert-evaluation-assets/working'
ABLATION = DATASET_BASE if not os.path.exists(f'{WORKING}/ablation_results.json') else WORKING
print(f'ABLATION path resolves to: {ABLATION}')
print(f'File exists: {os.path.exists(f"{ABLATION}/ablation_results.json")}')

# 2. Inspect raw C_late scores
with open(f'{WORKING}/phase_c_all_results.json') as f:
    r = json.load(f)

print('\n--- C_late raw scores ---')
print(json.dumps(r['phase_c_full_and_covered']['C_late'], indent=2))

print('\n--- C_additive raw scores ---')
print(json.dumps(r['phase_c_full_and_covered']['C_additive'], indent=2))

print('\n--- prior_conditions_on_c_covered keys ---')
prior = r['prior_conditions_on_c_covered']
for condition, vals in prior.items():
    print(f'{condition}: {vals}')

# 3. Inspect the ablation_results.json that Cell 27 will use
with open(f'{ABLATION}/ablation_results.json') as f:
    ab = json.load(f)

print('\n--- ablation_results.json top-level keys ---')
print(list(ab.keys()))

print('\n--- no_kg raw scores ---')
print(json.dumps(ab.get('no_kg', 'NOT FOUND'), indent=2))

print('\n--- A_late raw scores ---')
print(json.dumps(ab.get('A_late', 'NOT FOUND'), indent=2))

# 4. Cross-check the other ablation_results.json
OTHER = '/kaggle/input/datasets/michaelmarkey64/irish-ner-ablation-results/ablation_results.json'
if os.path.exists(OTHER):
    with open(OTHER) as f:
        ab2 = json.load(f)
    print('\n--- irish-ner-ablation-results top-level keys ---')
    print(list(ab2.keys()))
    print('\n--- no_kg raw scores (irish-ner-ablation-results) ---')
    print(json.dumps(ab2.get('no_kg', 'NOT FOUND'), indent=2))
else:
    print(f'\nirish-ner-ablation-results not found at {OTHER}')

ABLATION path resolves to: /kaggle/input/datasets/michaelmarkey64/gabert-evaluation-assets/working
File exists: True

--- C_late raw scores ---
{
  "42": {
    "full": 0.0137,
    "covered": 0.0159
  },
  "123": {
    "full": 0.0165,
    "covered": 0.0228
  },
  "256": {
    "full": 0.0174,
    "covered": 0.0231
  },
  "512": {
    "full": 0.024,
    "covered": 0.0367
  },
  "999": {
    "full": 0.0111,
    "covered": 0.016
  },
  "1024": {
    "full": 0.016,
    "covered": 0.0147
  },
  "2048": {
    "full": 0.0274,
    "covered": 0.0385
  }
}

--- C_additive raw scores ---
{
  "42": {
    "full": 0.0106,
    "covered": 0.0114
  },
  "123": {
    "full": 0.0252,
    "covered": 0.0411
  },
  "256": {
    "full": 0.016,
    "covered": 0.023
  },
  "512": {
    "full": 0.0172,
    "covered": 0.0198
  },
  "999": {
    "full": 0.014,
    "covered": 0.0147
  },
  "1024": {
    "full": 0.0176,
    "covered": 0.0241
  },
  "2048": {
    "full": 0.0134,
    "covered": 0.0188
  }
}

--- prior_

## CELL 28 — interpretation

### Results summary, mechanism explanation, and dissertation implications
### for Phase C. Written in third person following the established cell
### structure from Notebooks 02–04.

interpretation = """
### Phase C results: domain-matched KG augmentation

#### Results

[Insert results table from Cell 27 here after execution]

#### Mechanism

Phase C constructed a knowledge graph directly from entity surface strings
extracted from the Adkins et al. corpus, targeting maximum domain alignment.
Exhaustive search across Irish Wikipedia, English Wikipedia via Wikidata
sitelinks, GeoNames Ireland, the Oireachtas API, and Wikidata fuzzy search
yielded coverage of 34.1% (635 / 1863 unique entity surfaces). The remaining
65.9% of entities are absent from all structured public knowledge bases,
reflecting the long-tail character of the entity population in literary and
journalistic Irish text.

TransE embeddings (dim=128) were trained on 947 triples across 5 relation
types. The resulting embedding dict covers 638 head nodes; all other corpus
entities receive zero vectors at lookup time.

#### Dissertation implication

[Complete after Cell 27 results are available, following the pattern below
depending on outcome:]

IF null result persists (expected):
Phase C reproduces the null result under the highest coverage achievable
with public knowledge bases. Across both full-test and covered-subset
evaluations, KG augmentation does not improve on the no_kg baseline
(Wilcoxon p > 0.05 for all conditions). Combined with the Phase A and
Phase B null results, this constitutes strong evidence that TransE
embeddings do not provide a viable augmentation signal for token-level
Irish NER in this architecture. The coverage ceiling identified in the
systematic KB audit (34.1%) was a secondary rather than primary
explanatory mechanism: even on the 34.1% of entities for which KG
vectors were non-zero, augmentation did not improve performance.

IF C outperforms A/B (less likely):
Phase C shows improvement over Phases A and B on the covered subset,
suggesting that domain alignment — constructing the KG from corpus
entities rather than a pre-existing political domain graph — is a
meaningful factor in KG augmentation effectiveness. However, the
improvement over no_kg remains non-significant, indicating that the
mechanism provides signal relative to a misaligned KG but not relative
to no KG signal.
